# MindReader Biotech: Full Trial and Document Discovery

This notebook discovers public evidence for a clinical asset, drug, or trial across registry, literature, chemical, metadata, web, and PDF sources. It is designed for oncology and autoimmune assets, but the inputs are generic.

**Important retrieval limitation**

We retrieve all discoverable public, open-access, and user-provided documents from configured sources. Paywalled, private, unpublished, removed, or inaccessible material may not be retrievable.

**Why PubMed is not enough**

PubMed is essential, but it mainly catches title, abstract, and indexed metadata. It may miss papers where the asset appears only in body text, tables, figures, captions, supplementary files, PDFs, company pages, conference pages, investor decks, or registry result pages. This notebook therefore searches PubMed and then expands into PMC/BioC full text, Europe PMC, ClinicalTrials.gov, ChEMBL, PubChem, Crossref, OpenAlex, Semantic Scholar, Exa/web, pages, and PDFs.

No backend, Pydantic, FastAPI, or database is used. Everything is notebook-first and pandas-based.


## 1. Setup and imports

This section imports the standard analysis stack, optional parsers, and creates the output directory. The `safe_get` helper uses browser-like headers, retries transient errors, returns JSON or text, and never stops the whole notebook because one source fails.


In [1]:
import os
import re
import json
import time
import textwrap
import hashlib
import unicodedata
import xml.etree.ElementTree as ET
from pathlib import Path
from datetime import datetime
from urllib.parse import quote_plus, urlparse, urlunparse

import requests
import pandas as pd
import numpy as np

try:
    from bs4 import BeautifulSoup
except Exception:
    BeautifulSoup = None

try:
    import fitz  # PyMuPDF
except Exception:
    fitz = None

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = lambda x, **kwargs: x

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 220)
pd.set_option("display.width", 180)

OUTPUT_DIR = Path("discovery_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

RUN_STARTED_AT = datetime.utcnow().isoformat(timespec="seconds") + "Z"
SOURCE_STATUS = []


def log_source_status(source, searched=False, skipped=False, reason="", records=0):
    SOURCE_STATUS.append({
        "source": source,
        "searched": bool(searched),
        "skipped": bool(skipped),
        "reason": reason,
        "records": int(records) if records is not None else 0,
        "timestamp_utc": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    })


def safe_get(url, params=None, headers=None, timeout=30, retries=2, sleep=1.0):
    default_headers = {
        "User-Agent": "MindReaderBiotechDiscoveryNotebook/1.0 (+https://mindreader.bio; research; contact: notebook-user)",
        "Accept": "application/json,text/html,application/xml,text/plain,*/*",
    }
    merged_headers = {**default_headers, **(headers or {})}
    last_error = None
    for attempt in range(retries + 1):
        try:
            response = requests.get(url, params=params, headers=merged_headers, timeout=timeout)
            if response.status_code in {429, 500, 502, 503, 504} and attempt < retries:
                time.sleep(sleep * (attempt + 1))
                continue
            if not response.ok:
                print(f"[warn] GET {response.url} -> HTTP {response.status_code}")
                return None
            content_type = response.headers.get("content-type", "").lower()
            if "json" in content_type:
                try:
                    return response.json()
                except Exception as exc:
                    print(f"[warn] JSON parse failed for {response.url}: {exc}")
                    return None
            return response.text
        except Exception as exc:
            last_error = exc
            if attempt < retries:
                time.sleep(sleep * (attempt + 1))
    print(f"[warn] GET {url} failed: {last_error}")
    return None


def as_list(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple) or isinstance(value, set):
        return list(value)
    return [value]


def compact_join(values, sep="; "):
    clean = []
    for value in as_list(values):
        if value is None:
            continue
        text = str(value).strip()
        if text and text not in clean:
            clean.append(text)
    return sep.join(clean)


def first_nonempty(*values):
    for value in values:
        if value is None:
            continue
        if isinstance(value, float) and np.isnan(value):
            continue
        text = str(value).strip()
        if text:
            return text
    return ""


def empty_df(columns):
    return pd.DataFrame(columns=columns)


## 2. Source configuration

Toggles let the notebook run in constrained environments. API keys are loaded from environment variables and missing keys simply disable or reduce enrichment.


In [2]:
ASSET_NAME = "BEN-2293"
SPONSOR = "BenevolentAI"
DISEASE = "atopic dermatitis"
NCT_IDS = []
TARGETS = ["TrkA", "pan-Trk", "neurotrophin"]
KNOWN_PMIDS = []
KNOWN_DOIS = []
KNOWN_URLS = []
COMPANY_DOMAIN = ""  # optional, e.g. "benevolent.com"

USE_CT_GOV = True
USE_PUBMED = True
USE_PMC_BIOC = True
USE_EUROPE_PMC = True
USE_CHEMBL = True
USE_PUBCHEM = True
USE_CROSSREF = True
USE_OPENALEX = True
USE_SEMANTIC_SCHOLAR = True
USE_EXA = False  # enable only if EXA_API_KEY is available
USE_PDF_FETCH = True

EXA_API_KEY = os.getenv("EXA_API_KEY")
SEMANTIC_SCHOLAR_API_KEY = os.getenv("SEMANTIC_SCHOLAR_API_KEY")
NCBI_API_KEY = os.getenv("NCBI_API_KEY")
NCBI_EMAIL = os.getenv("NCBI_EMAIL", "your-email@example.com")

if USE_EXA and not EXA_API_KEY:
    print("[info] USE_EXA=True but EXA_API_KEY is missing; Exa calls will be skipped and Exa query bank will still be created.")

config_df = pd.DataFrame([{
    "asset": ASSET_NAME,
    "sponsor": SPONSOR,
    "disease": DISEASE,
    "nct_ids": NCT_IDS,
    "targets": TARGETS,
    "known_pmids": KNOWN_PMIDS,
    "known_dois": KNOWN_DOIS,
    "known_urls": KNOWN_URLS,
    "company_domain": COMPANY_DOMAIN,
    "run_started_at": RUN_STARTED_AT,
}])
display(config_df)


,asset,sponsor,disease,nct_ids,targets,known_pmids,known_dois,known_urls,company_domain,run_started_at
0,BEN-2293,BenevolentAI,atopic dermatitis,[],"[TrkA, pan-Trk, neurotrophin]",[],[],[],,2026-06-30T13:41:45Z


## 3. Alias generation

Aliases are deterministic and conservative. Source-derived aliases from registries and chemical databases are appended later with their provenance.


In [3]:
UNICODE_HYPHENS = ["-", "‑", "–", "—", "−", "‐"]


def normalize_unicode_hyphens(text):
    if text is None:
        return ""
    text = str(text).replace("\u00ad", "")
    for hyphen in UNICODE_HYPHENS:
        text = text.replace(hyphen, "-")
    return text


def normalize_ascii(text):
    return unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode("ascii")


def generate_deterministic_aliases(asset_name: str) -> set[str]:
    aliases = set()
    if not asset_name:
        return aliases
    original = str(asset_name).strip()
    base = re.sub(r"\s+", " ", original)
    variants = {original, base, base.lower(), base.upper(), base.title(), normalize_ascii(base)}
    no_parens = re.sub(r"\([^)]*\)", "", base).strip()
    if no_parens:
        variants.add(no_parens)
    for value in list(variants):
        clean = value.strip()
        if not clean:
            continue
        aliases.add(clean)
        aliases.add(re.sub(r"\s+", " ", clean))
        if "-" in normalize_unicode_hyphens(clean):
            normalized = normalize_unicode_hyphens(clean)
            pieces = [p for p in normalized.split("-") if p]
            if len(pieces) >= 2:
                aliases.add("".join(pieces))
                aliases.add(" ".join(pieces))
                aliases.add("_".join(pieces))
                aliases.add("-\n".join(pieces))
                aliases.add(" \n".join(pieces))
                aliases.add("\n".join(pieces))
                for hyphen in UNICODE_HYPHENS:
                    aliases.add(hyphen.join(pieces))
        if "/" in clean:
            aliases.add(clean.replace("/", " "))
            aliases.add(clean.replace("/", "-"))
        aliases.add(re.sub(r"\s+", "", clean))
    return {a for a in aliases if a and len(a.strip()) >= 2}


def quote_query(term: str) -> str:
    return f'"{term}"'


base_aliases = sorted(generate_deterministic_aliases(ASSET_NAME), key=lambda x: (len(x), x.lower()))
aliases_df = pd.DataFrame([{
    "alias": alias,
    "alias_type": "deterministic",
    "source": "input",
    "confidence": 90 if alias == ASSET_NAME else 75,
    "notes": "Generated from asset name spelling, hyphen, whitespace, case, ASCII, and PDF line-break variants.",
} for alias in base_aliases])

display(aliases_df.head(30))


,alias,alias_type,source,confidence,notes
0,Ben2293,deterministic,input,75,"Generated from asset name spelling, hyphen, whitespace, case, ASCII, and PDF line-break variants."
1,BEN2293,deterministic,input,75,"Generated from asset name spelling, hyphen, whitespace, case, ASCII, and PDF line-break variants."
2,ben2293,deterministic,input,75,"Generated from asset name spelling, hyphen, whitespace, case, ASCII, and PDF line-break variants."
3,BEN\n2293,deterministic,input,75,"Generated from asset name spelling, hyphen, whitespace, case, ASCII, and PDF line-break variants."
4,Ben\n2293,deterministic,input,75,"Generated from asset name spelling, hyphen, whitespace, case, ASCII, and PDF line-break variants."
5,ben\n2293,deterministic,input,75,"Generated from asset name spelling, hyphen, whitespace, case, ASCII, and PDF line-break variants."
6,BEN 2293,deterministic,input,75,"Generated from asset name spelling, hyphen, whitespace, case, ASCII, and PDF line-break variants."
7,ben 2293,deterministic,input,75,"Generated from asset name spelling, hyphen, whitespace, case, ASCII, and PDF line-break variants."
8,Ben 2293,deterministic,input,75,"Generated from asset name spelling, hyphen, whitespace, case, ASCII, and PDF line-break variants."
9,ben-2293,deterministic,input,75,"Generated from asset name spelling, hyphen, whitespace, case, ASCII, and PDF line-break variants."


## 4. ClinicalTrials.gov enrichment

ClinicalTrials.gov provides trial records, intervention names, other names, secondary IDs, registry references, and source-derived aliases. The search combines asset, sponsor, disease, target, and optional NCT IDs.


In [4]:
def fetch_ctgov_version():
    url = "https://clinicaltrials.gov/api/v2/version"
    data = safe_get(url)
    if isinstance(data, dict):
        return {
            "api_version": data.get("apiVersion") or data.get("version"),
            "data_timestamp": data.get("dataTimestamp"),
            "raw": data,
        }
    return {"api_version": None, "data_timestamp": None, "raw": data}


def fetch_ctgov_study(nct_id: str) -> dict | None:
    if not nct_id:
        return None
    url = f"https://clinicaltrials.gov/api/v2/studies/{nct_id.strip()}"
    data = safe_get(url)
    return data if isinstance(data, dict) else None


def search_ctgov_by_asset(asset, sponsor=None, disease=None, page_size=20):
    url = "https://clinicaltrials.gov/api/v2/studies"
    searches = []
    if asset:
        searches.append({"query.intr": asset, "query.term": asset, "label": "asset only"})
    if asset and sponsor:
        searches.append({"query.intr": asset, "query.spons": sponsor, "query.term": f"{asset} {sponsor}", "label": "asset + sponsor"})
    if asset and disease:
        searches.append({"query.intr": asset, "query.cond": disease, "query.term": f"{asset} {disease}", "label": "asset + disease"})
    if sponsor and disease:
        searches.append({"query.spons": sponsor, "query.cond": disease, "query.term": f"{sponsor} {disease}", "label": "sponsor + disease"})
    for target in TARGETS:
        searches.append({"query.cond": disease, "query.term": f"{target} {disease}", "label": f"target + disease: {target}"})

    studies = []
    seen = set()
    for spec in searches:
        params = {k: v for k, v in spec.items() if k != "label" and v}
        params.update({"pageSize": page_size, "format": "json"})
        data = safe_get(url, params=params)
        if not isinstance(data, dict):
            continue
        for study in data.get("studies", []):
            nct = (((study.get("protocolSection") or {}).get("identificationModule") or {}).get("nctId"))
            key = nct or json.dumps(study, sort_keys=True)[:200]
            if key in seen:
                continue
            seen.add(key)
            study["_query_used"] = spec.get("label")
            studies.append(study)
    return studies


def parse_ctgov_study(study):
    ps = study.get("protocolSection") or {}
    derived = study.get("derivedSection") or {}
    ident = ps.get("identificationModule") or {}
    status = ps.get("statusModule") or {}
    sponsor_mod = ps.get("sponsorCollaboratorsModule") or {}
    cond_mod = ps.get("conditionsModule") or {}
    design = ps.get("designModule") or {}
    arms_mod = ps.get("armsInterventionsModule") or {}
    outcomes = ps.get("outcomesModule") or {}
    refs_mod = ps.get("referencesModule") or {}
    results = study.get("resultsSection") or {}

    interventions = arms_mod.get("interventions") or []
    arms = arms_mod.get("armGroups") or []
    primary = outcomes.get("primaryOutcomes") or []
    secondary = outcomes.get("secondaryOutcomes") or []
    org = sponsor_mod.get("leadSponsor") or {}
    collaborators = sponsor_mod.get("collaborators") or []
    references = refs_mod.get("references") or []
    pubs = refs_mod.get("seeAlsoLinks") or []
    secondary_ids = ident.get("orgStudyIdInfo", {}).get("id")
    if isinstance(secondary_ids, str):
        secondary_ids = [secondary_ids]
    secondary_ids = as_list(secondary_ids) + [x.get("id") for x in ident.get("secondaryIdInfos", []) if isinstance(x, dict)]

    intervention_names = []
    other_names = []
    for intervention in interventions:
        intervention_names.append(intervention.get("name"))
        other_names.extend(intervention.get("otherNames") or [])

    row = {
        "nct_id": ident.get("nctId"),
        "brief_title": ident.get("briefTitle"),
        "official_title": ident.get("officialTitle"),
        "sponsor": org.get("name"),
        "collaborators": compact_join([c.get("name") for c in collaborators if isinstance(c, dict)]),
        "conditions": compact_join(cond_mod.get("conditions")),
        "interventions": compact_join(intervention_names),
        "intervention_other_names": compact_join(other_names),
        "arms": compact_join([a.get("label") for a in arms if isinstance(a, dict)]),
        "primary_outcomes": compact_join([o.get("measure") for o in primary if isinstance(o, dict)]),
        "secondary_outcomes": compact_join([o.get("measure") for o in secondary if isinstance(o, dict)]),
        "phase": compact_join(design.get("phases")),
        "status": status.get("overallStatus"),
        "start_date": (status.get("startDateStruct") or {}).get("date"),
        "completion_date": (status.get("completionDateStruct") or {}).get("date"),
        "last_update_posted": (status.get("lastUpdatePostDateStruct") or {}).get("date"),
        "has_results": bool(results),
        "references": len(references),
        "publications": len(pubs),
        "secondary_ids": compact_join(secondary_ids),
        "query_used": study.get("_query_used", ""),
        "url": f"https://clinicaltrials.gov/study/{ident.get('nctId')}" if ident.get("nctId") else "",
    }
    ref_rows = []
    for ref in references:
        if not isinstance(ref, dict):
            continue
        ref_rows.append({
            "source": "ClinicalTrials.gov",
            "nct_id": row["nct_id"],
            "pmid": ref.get("pmid"),
            "citation": ref.get("citation"),
            "url": f"https://pubmed.ncbi.nlm.nih.gov/{ref.get('pmid')}/" if ref.get("pmid") else row["url"],
            "type": "reference",
        })
    for pub in pubs:
        if not isinstance(pub, dict):
            continue
        ref_rows.append({
            "source": "ClinicalTrials.gov",
            "nct_id": row["nct_id"],
            "pmid": "",
            "citation": pub.get("label"),
            "url": pub.get("url") or row["url"],
            "type": "see_also",
        })
    alias_values = []
    alias_values.extend(intervention_names)
    alias_values.extend(other_names)
    alias_values.extend(secondary_ids)
    alias_values.append(row["sponsor"])
    alias_values.extend(as_list(cond_mod.get("conditions")))
    for title in [row["brief_title"], row["official_title"]]:
        if title:
            alias_values.extend(re.findall(r"\b[A-Z]{2,}[-\s]?\d{2,}\b", title))
    alias_rows = [{
        "alias": str(a).strip(),
        "alias_type": "source_derived",
        "source": "ClinicalTrials.gov",
        "confidence": 80,
        "notes": f"Derived from CT.gov study {row['nct_id']}",
    } for a in alias_values if a and str(a).strip()]
    return row, ref_rows, alias_rows


ctgov_trials_df = empty_df(["nct_id","brief_title","official_title","sponsor","collaborators","conditions","interventions","intervention_other_names","arms","primary_outcomes","secondary_outcomes","phase","status","start_date","completion_date","last_update_posted","has_results","references","publications","secondary_ids","query_used","url"])
ctgov_refs_df = empty_df(["source","nct_id","pmid","citation","url","type"])
ctgov_aliases_df = empty_df(["alias","alias_type","source","confidence","notes"])
ctgov_version = {}

if USE_CT_GOV:
    ctgov_version = fetch_ctgov_version()
    studies = []
    for nct_id in NCT_IDS:
        study = fetch_ctgov_study(nct_id)
        if study:
            study["_query_used"] = "input NCT"
            studies.append(study)
    studies.extend(search_ctgov_by_asset(ASSET_NAME, SPONSOR, DISEASE, page_size=20))
    rows, ref_rows, alias_rows = [], [], []
    for study in studies:
        row, refs, aliases = parse_ctgov_study(study)
        rows.append(row)
        ref_rows.extend(refs)
        alias_rows.extend(aliases)
    if rows:
        ctgov_trials_df = pd.DataFrame(rows).drop_duplicates(subset=["nct_id"], keep="first")
    if ref_rows:
        ctgov_refs_df = pd.DataFrame(ref_rows).drop_duplicates()
    if alias_rows:
        ctgov_aliases_df = pd.DataFrame(alias_rows).drop_duplicates(subset=["alias","source"])
    log_source_status("ClinicalTrials.gov", searched=True, records=len(ctgov_trials_df), reason=f"version={ctgov_version.get('api_version')}, timestamp={ctgov_version.get('data_timestamp')}")
else:
    log_source_status("ClinicalTrials.gov", skipped=True, reason="USE_CT_GOV=False")

aliases_df = pd.concat([aliases_df, ctgov_aliases_df], ignore_index=True).drop_duplicates(subset=["alias","source"], keep="first")
display(ctgov_trials_df.head())
display(ctgov_refs_df.head())
display(ctgov_aliases_df.head())


,nct_id,brief_title,official_title,sponsor,collaborators,conditions,interventions,intervention_other_names,arms,primary_outcomes,secondary_outcomes,phase,status,start_date,completion_date,last_update_posted,has_results,references,publications,secondary_ids,query_used,url
0,NCT04737304,A First-in-Human PoC Study With BEN2293 in Patients With Mild to Moderate Atopic Dermatitis,"A First-in-Human, Double-Blind, Randomised, Vehicle Controlled Phase I/II Proof of Concept Study to Investigate the Safety, Tolerability, Pharmacokinetics and Efficacy of BEN2293 in Patients With Mild to Moderate Ato...",BenevolentAI Bio,,Atopic Dermatitis,BEN2293 (0.25% or 1.0% w/w) or matching placebo,BEN2293,Dose Regimen Low Dose; Dose Regimen High Dose; Placebo,"Safety and tolerability assessed by means of incidence of adverse events, incidence of adverse events at the local application site, mean vital signs, mean 12-lead ECG parameters and mean safety laboratory results.",PK-Cmax; PK-Tmax; PK-T1/2; PK-AUC; PK- over a dosing interval (AUCт); PK - Accumulation ratio; Time to itch reduction; Fraction of patients achieving itch reduction; Change from baseline in the Numerical Rating Scale...,PHASE1; PHASE2,COMPLETED,2020-10-14,2023-01-26,2023-06-18,False,0,0,BB-2293-101b,asset only,https://clinicaltrials.gov/study/NCT04737304
1,NCT04986384,Effect of a Topical Spray on Itch Relief in Moderate-to-severe Childhood Eczema,Effect of a Topical Spray on Itch Relief in Moderate-to-severe Childhood Eczema,Prof. HON Kam Lun Ellis,,Eczema,Atoderm SOS spray Aerosol 200ml,,Active Treatment Group; Wait-list Control Group,SCORing Atopic Dermatitis (SCORAD); The Patient Oriented Eczema Measure (POEM); Nottingham Eczema Severity Score (NESS); The Children's Dermatology Life Quality Index (CDLQI); Pediatric Allergic Disease Quality of Li...,Bacterial colonization; dermatological parameters on skin; seromarkers,NA,COMPLETED,2019-01-24,2021-04-07,2021-08-02,False,0,0,BAIS2018,target + disease: neurotrophin,https://clinicaltrials.gov/study/NCT04986384
2,NCT00557284,Efficacy Study of Montelukast in Atopic Dermatitis Induced by Food Allergens,"A Randomized, Double-blind, Placebo-controlled, Parallel Group Study to Evaluate the Efficacy of Montelukast (Singulair) in Participants Ages 1 - 8 Years Diagnosed With Atopic Dermatitis Induced by Food Allergens",1st Allergy & Clinical Research Center,,Atopic Dermatitis,Montelukast; Placebo,,1; 2,Change in Percentage of Body Involvement; Mean Change in Investigator Global Assessment (IGA); Mean Change in PADC (Caregivers Perception of Disease Control); Mean Change in Pruritus; Mean Change in Weekly Use of Res...,Mean Change in Serum and Urinary Inflammatory Marker Levels; Mean Change in Serum IgE Levels; Mean Change in (Gastrointestinal Symptom Rating Scale) GSRS,PHASE4,COMPLETED,2008-03,2009-05,2015-04-21,True,0,1,32032,target + disease: neurotrophin,https://clinicaltrials.gov/study/NCT00557284
3,NCT03831646,"Clinical, Psychological and Genetic Characteristics in Dermatological Patients","Clinical, Psychological and Genetic Characteristics of Patients With Atopic Dermatitis and Psoriasis",Astana Medical University,"Ariel University; Tirat Carmel Mental Health Center; Medical Centre Hospital of the President's Affairs Administration, Republic of Kazakhstan",Atopic Dermatitis; Psoriasis,,,dermatological patients,Assessment of change in the severity of atopic dermatitis after conventional treatment from study onset (baseline) at week 10; Assessment of change in the severity of psoriasis after conventional treatment from study...,"Assessment and comparison (Unpaired t-test) of SCORAD scores in extrinsic atopic dermatitis (EAD, IgE level above the normal) and intrinsic atopic dermatitis (IAD, normal IgE level) patients compared with baseline af...",,COMPLETED,2019-01-20,2019-07-30,2019-08-20,False,0,10,4,target + disease: neurotrophin,https://clinicaltrials.gov/study/NCT03831646


,source,nct_id,pmid,citation,url,type
0,ClinicalTrials.gov,NCT00557284,,Research website,http://www.immunoeresearch.com,see_also
1,ClinicalTrials.gov,NCT03831646,,Association of HPA axis hormones with copeptin after psychological stress differs by sex,http://www.ncbi.nlm.nih.gov/pubmed/26520685,see_also
2,ClinicalTrials.gov,NCT03831646,,How does epidermal pathology interact with mental state?,http://www.ncbi.nlm.nih.gov/pubmed/23245205,see_also
3,ClinicalTrials.gov,NCT03831646,,The roles of BDNF in the pathophysiology of major depression and in antidepressant treatment,http://www.ncbi.nlm.nih.gov/pubmed/21253405,see_also
4,ClinicalTrials.gov,NCT03831646,,Increase in serum brain-derived neurotrophic factor in met allele carriers of the BDNF Val66Met polymorphism is specific to males,http://www.ncbi.nlm.nih.gov/pubmed/22538247,see_also


,alias,alias_type,source,confidence,notes
0,BEN2293 (0.25% or 1.0% w/w) or matching placebo,source_derived,ClinicalTrials.gov,80,Derived from CT.gov study NCT04737304
1,BEN2293,source_derived,ClinicalTrials.gov,80,Derived from CT.gov study NCT04737304
2,BB-2293-101b,source_derived,ClinicalTrials.gov,80,Derived from CT.gov study NCT04737304
3,BenevolentAI Bio,source_derived,ClinicalTrials.gov,80,Derived from CT.gov study NCT04737304
4,Atopic Dermatitis,source_derived,ClinicalTrials.gov,80,Derived from CT.gov study NCT04737304


## 5. Query bank builder

The query bank intentionally separates exact asset queries from sponsor/disease, mechanism, registry, PDF, conference, company, and SEC queries. These rows drive API searches and also serve as a manual search plan when a source is disabled.


In [5]:
def build_query_bank(asset_aliases, sponsor, disease, targets, nct_ids):
    rows = []

    def add(query, category, priority, expected_source, notes=""):
        if query and query not in {r["query"] for r in rows}:
            rows.append({
                "query": query,
                "category": category,
                "priority": priority,
                "expected_source": expected_source,
                "notes": notes,
            })

    def is_searchable_asset_alias(value):
        text = str(value or "").strip()
        if not text or "\n" in text or len(text) > 80:
            return False
        normalized_value = re.sub(r"[^a-z0-9]", "", normalize_ascii(normalize_unicode_hyphens(text)).lower())
        normalized_asset = re.sub(r"[^a-z0-9]", "", normalize_ascii(normalize_unicode_hyphens(ASSET_NAME)).lower())
        has_letters_and_digits = bool(re.search(r"[A-Za-z]", text) and re.search(r"\d", text))
        asset_digits = "".join(re.findall(r"\d+", ASSET_NAME))
        contains_asset_code = normalized_asset and normalized_asset in normalized_value
        contains_asset_digits = bool(asset_digits and asset_digits in normalized_value and has_letters_and_digits)
        return normalized_value == normalized_asset or contains_asset_code or contains_asset_digits

    preferred_aliases = []
    for a in asset_aliases:
        if is_searchable_asset_alias(a):
            preferred_aliases.append(str(a).strip())
    if not preferred_aliases:
        preferred_aliases = [ASSET_NAME]
    preferred_aliases = sorted(set(preferred_aliases), key=lambda x: (0 if x == ASSET_NAME else 1, len(x), x.lower()))[:12]

    for alias in preferred_aliases:
        q = quote_query(alias)
        add(q, "exact_asset", 1, "all", "Exact asset alias")
        if disease:
            add(f'{q} {quote_query(disease)}', "asset_disease", 1, "literature/web/registry", "Asset plus disease")
        if sponsor:
            add(f'{q} {quote_query(sponsor)}', "asset_sponsor", 1, "web/company/registry", "Asset plus sponsor")
        add(f'{q} filetype:pdf', "file_pdf", 2, "web/pdf", "PDF search")
        if sponsor:
            add(f'{quote_query(sponsor)} {q} filetype:pdf', "file_pdf", 2, "web/pdf", "Sponsor PDF search")
        for term in ["poster", "presentation", "investor deck", "conference abstract", "results", "discontinued", "phase"]:
            add(f'{q} "{term}"', "file_pdf", 2, "web/conference/company", f"Asset plus {term}")

    if sponsor and disease:
        for term in ["trial", "phase", "discontinued", "results"]:
            add(f'{quote_query(sponsor)} {quote_query(disease)} {term}', "sponsor_disease", 2, "web/registry", f"Sponsor disease {term}")

    for target in targets or []:
        if disease:
            add(f'{quote_query(target)} {quote_query(disease)}', "target_mechanism", 3, "literature/web", "Target plus disease")
        add(f'{quote_query(str(target) + " inhibitor")} {quote_query(disease)}' if disease else quote_query(str(target) + " inhibitor"), "target_mechanism", 3, "literature/web", "Inhibitor mechanism")
    if disease:
        add(f'{quote_query("neurotrophin")} {quote_query("pruritus")}', "target_mechanism", 3, "literature/web", "Neurotrophin pruritus context")

    for nct_id in nct_ids or []:
        add(quote_query(nct_id), "nct", 1, "registry/literature/web", "Known NCT ID")

    conference_domains = {
        "ASCO": "asco.org",
        "ESMO": "esmo.org",
        "AACR": "aacr.org",
        "SITC": "sitcancer.org",
        "EHA": "ehaweb.org",
        "ASH": "ashpublications.org",
        "ACR": "acrabstracts.org",
        "EULAR": "eular.org",
        "EADV": "eadv.org",
        "DDW": "ddw.org",
        "UEG": "ueg.eu",
        "ADA": "diabetes.org",
        "ENDO": "endocrine.org",
    }
    for alias in preferred_aliases[:5]:
        for society, domain in conference_domains.items():
            add(f'site:{domain} {quote_query(alias)}', "conference_society", 4, society, f"{society} domain search")
        if sponsor:
            add(f'site:sec.gov {quote_query(alias)} {quote_query(sponsor)}', "sec_company", 3, "SEC", "SEC/company filings")
        if COMPANY_DOMAIN:
            add(f'site:{COMPANY_DOMAIN} {quote_query(alias)}', "company_site", 2, "company", "Company domain search")

    return pd.DataFrame(rows).sort_values(["priority", "category", "query"]).reset_index(drop=True)


query_bank_df = build_query_bank(aliases_df["alias"].dropna().tolist(), SPONSOR, DISEASE, TARGETS, list(set(NCT_IDS + ctgov_trials_df.get("nct_id", pd.Series(dtype=str)).dropna().tolist())))
exa_query_bank_df = query_bank_df[query_bank_df["expected_source"].str.contains("web|pdf|company|conference|SEC|all", case=False, na=False)].copy()
display(query_bank_df.head(50))


,query,category,priority,expected_source,notes
0,"""32032"" ""atopic dermatitis""",asset_disease,1,literature/web/registry,Asset plus disease
1,"""4"" ""atopic dermatitis""",asset_disease,1,literature/web/registry,Asset plus disease
2,"""BAIS2018"" ""atopic dermatitis""",asset_disease,1,literature/web/registry,Asset plus disease
3,"""BEN 2293"" ""atopic dermatitis""",asset_disease,1,literature/web/registry,Asset plus disease
4,"""BEN-2293"" ""atopic dermatitis""",asset_disease,1,literature/web/registry,Asset plus disease
5,"""BEN2293"" ""atopic dermatitis""",asset_disease,1,literature/web/registry,Asset plus disease
6,"""Ben 2293"" ""atopic dermatitis""",asset_disease,1,literature/web/registry,Asset plus disease
7,"""Ben2293"" ""atopic dermatitis""",asset_disease,1,literature/web/registry,Asset plus disease
8,"""Eczema"" ""atopic dermatitis""",asset_disease,1,literature/web/registry,Asset plus disease
9,"""Placebo"" ""atopic dermatitis""",asset_disease,1,literature/web/registry,Asset plus disease


## 6. PubMed search

PubMed is searched through NCBI E-utilities. Results preserve the query used, identifiers, title, abstract, DOI, PMCID, journal, publication date, authors, and compact match snippets.


In [7]:
def ncbi_params(extra):
    params = dict(extra)
    if NCBI_API_KEY:
        params["api_key"] = NCBI_API_KEY
    if NCBI_EMAIL:
        params["email"] = NCBI_EMAIL
        params["tool"] = "mindreader_discovery_notebook"
    return params


def pubmed_esearch(query, retmax=50):
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = ncbi_params({"db": "pubmed", "term": query, "retmode": "json", "retmax": retmax})
    data = safe_get(url, params=params)
    if isinstance(data, dict):
        return data.get("esearchresult", {}).get("idlist", [])
    return []


def pubmed_esummary(pmids):
    pmids = [str(p) for p in as_list(pmids) if p]
    if not pmids:
        return {}
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"
    params = ncbi_params({"db": "pubmed", "id": ",".join(pmids), "retmode": "json"})
    data = safe_get(url, params=params)
    return data if isinstance(data, dict) else {}


def pubmed_efetch_xml(pmids):
    pmids = [str(p) for p in as_list(pmids) if p]
    if not pmids:
        return None
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    params = ncbi_params({"db": "pubmed", "id": ",".join(pmids), "retmode": "xml"})
    data = safe_get(url, params=params)
    if isinstance(data, str):
        try:
            return ET.fromstring(data)
        except Exception as exc:
            print(f"[warn] PubMed XML parse failed: {exc}")
    return None


def text_from_nodes(nodes):
    parts = []
    for node in nodes:
        text = "".join(node.itertext()).strip()
        if text:
            parts.append(text)
    return "\n".join(parts)


def parse_pubmed_xml(root, query_by_pmid=None):
    rows = []
    query_by_pmid = query_by_pmid or {}
    if root is None:
        return rows
    for article in root.findall(".//PubmedArticle"):
        medline = article.find("./MedlineCitation")
        pmid = medline.findtext("./PMID") if medline is not None else ""
        art = medline.find("./Article") if medline is not None else None
        title = text_from_nodes(art.findall("./ArticleTitle")) if art is not None else ""
        abstract = text_from_nodes(art.findall("./Abstract/AbstractText")) if art is not None else ""
        journal = art.findtext("./Journal/Title") if art is not None else ""
        pub_date = ""
        if art is not None:
            date_node = art.find("./Journal/JournalIssue/PubDate")
            if date_node is not None:
                pub_date = "-".join([date_node.findtext(x) or "" for x in ["Year","Month","Day"]]).strip("-")
        article_ids = {}
        for aid in article.findall("./PubmedData/ArticleIdList/ArticleId"):
            article_ids[aid.attrib.get("IdType", "")] = (aid.text or "").strip()
        pub_types = [x.text for x in article.findall(".//PublicationType") if x.text]
        authors = []
        for author in article.findall(".//AuthorList/Author"):
            last = author.findtext("LastName") or ""
            fore = author.findtext("ForeName") or ""
            collective = author.findtext("CollectiveName") or ""
            name = collective or " ".join([fore, last]).strip()
            if name:
                authors.append(name)
        rows.append({
            "source": "PubMed",
            "query_used": query_by_pmid.get(str(pmid), ""),
            "pmid": str(pmid),
            "pmcid": article_ids.get("pmc", ""),
            "doi": article_ids.get("doi", ""),
            "title": title,
            "abstract": abstract,
            "journal": journal,
            "pub_date": pub_date,
            "url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/" if pmid else "",
            "publication_types": compact_join(pub_types),
            "authors": compact_join(authors[:12]),
        })
    return rows


def alias_to_flexible_pattern(alias):
    alias_norm = normalize_unicode_hyphens(alias).replace("\n", " ").strip()
    if not alias_norm:
        return None
    tokens = [tok for tok in re.split(r"[\s_\-]+", alias_norm) if tok]
    if len(tokens) >= 2:
        separator = r"(?:[\s_\-])*"
        return separator.join(re.escape(tok) for tok in tokens)
    return re.escape(alias_norm)


def find_alias_matches(text, aliases, window=250):
    if not text:
        return []
    raw = str(text)
    normalized = normalize_unicode_hyphens(raw)
    compact_linebreak = re.sub(r"([A-Za-z]+)[-\s]*\n\s*([0-9]+)", r"\1-\2", normalized)
    searchable = compact_linebreak
    matches = []
    aliases_sorted = sorted({str(a).strip() for a in aliases if str(a).strip()}, key=len, reverse=True)
    for alias in aliases_sorted:
        pattern = alias_to_flexible_pattern(alias)
        if not pattern:
            continue
        try:
            iterator = re.finditer(pattern, searchable, flags=re.IGNORECASE)
        except re.error as exc:
            print(f"[warn] Skipping invalid alias regex for {alias!r}: {exc}")
            continue
        for match in iterator:
            start, end = match.span()
            snippet = searchable[max(0, start-window):min(len(searchable), end+window)]
            snippet = re.sub(r"\s+", " ", snippet).strip()
            matches.append({
                "alias_matched": alias,
                "match_context": snippet,
                "start": start,
                "end": end,
            })
            break
    return matches


pubmed_columns = ["source","query_used","pmid","pmcid","doi","title","abstract","journal","pub_date","url","match_location","alias_matched","match_context","confidence","publication_types","authors"]
pubmed_hits_df = empty_df(pubmed_columns)

if USE_PUBMED:
    pmid_to_query = {}
    high_priority_queries = query_bank_df[query_bank_df["priority"].le(2)]["query"].head(40).tolist()
    for known in KNOWN_PMIDS:
        pmid_to_query[str(known)] = "user_provided_pmid"
    for query in tqdm(high_priority_queries, desc="PubMed searches"):
        ids = pubmed_esearch(query, retmax=50)
        for pmid in ids:
            pmid_to_query.setdefault(str(pmid), query)
        time.sleep(0.12 if NCBI_API_KEY else 0.35)
    rows = []
    for chunk_start in range(0, len(pmid_to_query), 100):
        chunk = list(pmid_to_query.keys())[chunk_start:chunk_start+100]
        root = pubmed_efetch_xml(chunk)
        rows.extend(parse_pubmed_xml(root, pmid_to_query))
        time.sleep(0.12 if NCBI_API_KEY else 0.35)
    out_rows = []
    aliases = aliases_df["alias"].dropna().tolist()
    for row in rows:
        title_matches = find_alias_matches(row.get("title",""), aliases, window=180)
        abstract_matches = find_alias_matches(row.get("abstract",""), aliases, window=220)
        all_matches = [(m, "title") for m in title_matches] + [(m, "abstract") for m in abstract_matches]
        if not all_matches:
            all_matches = [({"alias_matched": "", "match_context": ""}, "metadata")]
        for match, loc in all_matches:
            new = dict(row)
            new.update({
                "match_location": loc,
                "alias_matched": match.get("alias_matched", ""),
                "match_context": match.get("match_context", ""),
                "confidence": 0,
            })
            out_rows.append(new)
    if out_rows:
        pubmed_hits_df = pd.DataFrame(out_rows).drop_duplicates(subset=["pmid","match_location","alias_matched","query_used"])
    log_source_status("PubMed", searched=True, records=len(pubmed_hits_df), reason=f"{len(high_priority_queries)} high-priority queries")
else:
    log_source_status("PubMed", skipped=True, reason="USE_PUBMED=False")

display(pubmed_hits_df.head())


PubMed searches:   0%|          | 0/40 [00:00<?, ?it/s]

,source,query_used,pmid,pmcid,doi,title,abstract,journal,pub_date,url,publication_types,authors,match_location,alias_matched,match_context,confidence
0,PubMed,"""4"" ""atopic dermatitis""",42374848,,10.1177/17103568261457273,Assessment and Clinical Management of Patients With Atopic Dermatitis Undergoing Patch Testing: Recommendations From an International Electronic Delphi Consensus.,"Individuals with atopic dermatitis (AD) may be referred for patch testing to rule out allergic contact dermatitis (ACD). While past expert consensus outlines when and how to patch test these patients, clinical manage...","Dermatitis : contact, atopic, occupational, drug",2026-Jun-30,https://pubmed.ncbi.nlm.nih.gov/42374848/,Journal Article,Jamie P Schlarbaum; Erin M Warshaw; Yael A Leshem; Jonathan I Silverberg; Sara Hylwa; Susan Nedorost; Aaron Drucker; Lisa A Beck; Peggy A Wu; Jennifer K Chen; JiaDe Yu; Brandon L Adler,title,Atopic Dermatitis,Assessment and Clinical Management of Patients With Atopic Dermatitis Undergoing Patch Testing: Recommendations From an International Electronic Delphi Consensus.,0
1,PubMed,"""4"" ""atopic dermatitis""",42374848,,10.1177/17103568261457273,Assessment and Clinical Management of Patients With Atopic Dermatitis Undergoing Patch Testing: Recommendations From an International Electronic Delphi Consensus.,"Individuals with atopic dermatitis (AD) may be referred for patch testing to rule out allergic contact dermatitis (ACD). While past expert consensus outlines when and how to patch test these patients, clinical manage...","Dermatitis : contact, atopic, occupational, drug",2026-Jun-30,https://pubmed.ncbi.nlm.nih.gov/42374848/,Journal Article,Jamie P Schlarbaum; Erin M Warshaw; Yael A Leshem; Jonathan I Silverberg; Sara Hylwa; Susan Nedorost; Aaron Drucker; Lisa A Beck; Peggy A Wu; Jennifer K Chen; JiaDe Yu; Brandon L Adler,abstract,Atopic Dermatitis,"Individuals with atopic dermatitis (AD) may be referred for patch testing to rule out allergic contact dermatitis (ACD). While past expert consensus outlines when and how to patch test these patients, clinical manage...",0
2,PubMed,"""4"" ""atopic dermatitis""",42374848,,10.1177/17103568261457273,Assessment and Clinical Management of Patients With Atopic Dermatitis Undergoing Patch Testing: Recommendations From an International Electronic Delphi Consensus.,"Individuals with atopic dermatitis (AD) may be referred for patch testing to rule out allergic contact dermatitis (ACD). While past expert consensus outlines when and how to patch test these patients, clinical manage...","Dermatitis : contact, atopic, occupational, drug",2026-Jun-30,https://pubmed.ncbi.nlm.nih.gov/42374848/,Journal Article,Jamie P Schlarbaum; Erin M Warshaw; Yael A Leshem; Jonathan I Silverberg; Sara Hylwa; Susan Nedorost; Aaron Drucker; Lisa A Beck; Peggy A Wu; Jennifer K Chen; JiaDe Yu; Brandon L Adler,abstract,4,nical recommendations in the assessment and management of patients with AD undergoing patch testing. An international modified electronic (e)Delphi consensus exercise was conducted among 18 AD and ACD experts. Follow...,0
3,PubMed,"""4"" ""atopic dermatitis""",42371286,,10.1007/s40261-026-01573-9,"Efficacy, Safety, Quality of Life, and Adherence of Dupilumab in Pediatric Atopic Dermatitis: A Systematic Review.","Atopic dermatitis (AD) is a chronic, relapsing inflammatory skin disease that disproportionately affects children, often causing significant physical, psychological, and social burdens. While biologic therapies have ...",Clinical drug investigation,2026-Jun-29,https://pubmed.ncbi.nlm.nih.gov/42371286/,Journal Article; Systematic Review,Jacob Surma; Jenna Currier; David Doyle; Mackenzy Maddock; Garrett Borgman; Cassandra DeGroat; Sierra Demarinis; Natalie Aguilar; Spencer Williams; Patrick Fakhoury; Robert Stachler,title,Atopic Dermatitis,"Efficacy, Safety, Quality of Life, and Adherence of Dupilumab in Pediatric Atopic Dermatitis: A Systematic Review.",0
4,PubMed,"""4"" ""atop

## 7. PMC / BioC full-text retrieval and scanning

PMC Open Access BioC retrieval can expose body, table, and figure-caption mentions that PubMed title/abstract search may miss. The notebook stores only short match snippets, not full copyrighted text.


In [8]:
def fetch_pmc_bioc(identifier):
    identifier = str(identifier).strip()
    if not identifier:
        return None
    url = f"https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_json/{identifier}/unicode"
    data = safe_get(url, timeout=45)
    return data if isinstance(data, dict) else None


def flatten_bioc_passages(bioc_json):
    rows = []
    if not isinstance(bioc_json, dict):
        return rows
    documents = bioc_json.get("documents") or []
    for doc in documents:
        doc_id = doc.get("id", "")
        for passage in doc.get("passages", []) or []:
            infons = passage.get("infons") or {}
            section = first_nonempty(infons.get("section_type"), infons.get("type"), infons.get("section"), "unknown")
            text = passage.get("text") or ""
            offset = passage.get("offset")
            lower_section = str(section).lower()
            if "table" in lower_section:
                location = "table"
            elif "fig" in lower_section or "caption" in lower_section:
                location = "figure_caption"
            elif "abstract" in lower_section:
                location = "abstract"
            elif "title" in lower_section:
                location = "title"
            elif "ref" in lower_section:
                location = "reference"
            else:
                location = "body"
            rows.append({"doc_id": doc_id, "section": section, "match_location": location, "offset": offset, "text": text})
    return rows


pmc_columns = ["source","pmid","pmcid","title","section","match_location","alias_matched","match_context","url","confidence"]
pmc_fulltext_hits_df = empty_df(pmc_columns)

if USE_PMC_BIOC:
    identifiers = []
    for col in ["pmcid", "pmid"]:
        if col in pubmed_hits_df.columns:
            identifiers.extend(pubmed_hits_df[col].dropna().astype(str).tolist())
    identifiers = [x for x in dict.fromkeys(identifiers) if x and x.lower() != "nan"]
    rows = []
    aliases = aliases_df["alias"].dropna().tolist()
    for identifier in tqdm(identifiers[:150], desc="PMC BioC"):
        data = fetch_pmc_bioc(identifier)
        passages = flatten_bioc_passages(data)
        for passage in passages:
            matches = find_alias_matches(passage.get("text",""), aliases, window=250)
            for match in matches:
                rows.append({
                    "source": "PMC BioC",
                    "pmid": identifier if identifier.isdigit() else "",
                    "pmcid": identifier if identifier.upper().startswith("PMC") else "",
                    "title": "",
                    "section": passage.get("section"),
                    "match_location": passage.get("match_location"),
                    "alias_matched": match.get("alias_matched"),
                    "match_context": match.get("match_context"),
                    "url": f"https://www.ncbi.nlm.nih.gov/pmc/articles/{identifier}/" if identifier.upper().startswith("PMC") else f"https://pubmed.ncbi.nlm.nih.gov/{identifier}/",
                    "confidence": 0,
                })
        time.sleep(0.2)
    if rows:
        pmc_fulltext_hits_df = pd.DataFrame(rows).drop_duplicates()
    log_source_status("PMC BioC", searched=True, records=len(pmc_fulltext_hits_df), reason=f"{len(identifiers)} PubMed/PMCID identifiers attempted")
else:
    log_source_status("PMC BioC", skipped=True, reason="USE_PMC_BIOC=False")

display(pmc_fulltext_hits_df.head())


PMC BioC:   0%|          | 0/150 [00:00<?, ?it/s]

,source,pmid,pmcid,title,section,match_location,alias_matched,match_context,url,confidence


## 8. Europe PMC search

Europe PMC broadens literature discovery and can return open-access flags, full-text URLs, citation counts, abstracts, PMIDs, PMCIDs, and DOIs.


In [9]:
def europe_pmc_search(query, page_size=25):
    url = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
    params = {"query": query, "format": "json", "pageSize": page_size, "resultType": "core"}
    data = safe_get(url, params=params)
    if isinstance(data, dict):
        return data.get("resultList", {}).get("result", [])
    return []


europe_columns = ["source","query_used","title","authorString","journalTitle","pubYear","doi","pmid","pmcid","isOpenAccess","inEPMC","citedByCount","fullTextUrlList","abstractText","url","match_location","alias_matched","match_context","confidence"]
europe_pmc_hits_df = empty_df(europe_columns)

if USE_EUROPE_PMC:
    rows = []
    queries = query_bank_df[query_bank_df["priority"].le(2)]["query"].head(35).tolist()
    queries += [f'DOI:"{doi}"' for doi in KNOWN_DOIS]
    for query in tqdm(dict.fromkeys(queries), desc="Europe PMC"):
        for item in europe_pmc_search(query, page_size=25):
            full_urls = item.get("fullTextUrlList", {}).get("fullTextUrl", []) if isinstance(item.get("fullTextUrlList"), dict) else []
            url = item.get("doi") and f"https://doi.org/{item.get('doi')}" or item.get("pmid") and f"https://pubmed.ncbi.nlm.nih.gov/{item.get('pmid')}/" or ""
            text_blob = " ".join([item.get("title") or "", item.get("abstractText") or ""])
            matches = find_alias_matches(text_blob, aliases_df["alias"].dropna().tolist(), window=220)
            if not matches:
                matches = [{"alias_matched": "", "match_context": ""}]
            for match in matches:
                rows.append({
                    "source": "Europe PMC",
                    "query_used": query,
                    "title": item.get("title"),
                    "authorString": item.get("authorString"),
                    "journalTitle": item.get("journalTitle"),
                    "pubYear": item.get("pubYear"),
                    "doi": item.get("doi"),
                    "pmid": item.get("pmid"),
                    "pmcid": item.get("pmcid"),
                    "isOpenAccess": item.get("isOpenAccess"),
                    "inEPMC": item.get("inEPMC"),
                    "citedByCount": item.get("citedByCount"),
                    "fullTextUrlList": json.dumps(full_urls)[:1000],
                    "abstractText": item.get("abstractText"),
                    "url": url,
                    "match_location": "abstract" if match.get("alias_matched") and item.get("abstractText") and match.get("alias_matched","").lower() in item.get("abstractText","").lower() else "metadata",
                    "alias_matched": match.get("alias_matched"),
                    "match_context": match.get("match_context"),
                    "confidence": 0,
                })
        time.sleep(0.2)
    if rows:
        europe_pmc_hits_df = pd.DataFrame(rows).drop_duplicates(subset=["query_used","title","doi","pmid","pmcid","alias_matched"])
    log_source_status("Europe PMC", searched=True, records=len(europe_pmc_hits_df), reason=f"{len(queries)} queries")
else:
    log_source_status("Europe PMC", skipped=True, reason="USE_EUROPE_PMC=False")

display(europe_pmc_hits_df.head())


Europe PMC:   0%|          | 0/35 [00:00<?, ?it/s]

,source,query_used,title,authorString,journalTitle,pubYear,doi,pmid,pmcid,isOpenAccess,inEPMC,citedByCount,fullTextUrlList,abstractText,url,match_location,alias_matched,match_context,confidence
0,Europe PMC,"""32032"" ""atopic dermatitis""",Exploring the Impact of a Structured Educational Approach on Peristomal Skin Complications: An Interim Analysis.,"Denti FC, Guerra E, Caroppo F, Abruzzese P, Alessi F, Barone F, Bernardino P, Bergamini M, Bernardo C, Bosio G, Carp P, Cecconello M, Cerchier A, Croci F, Detti R, Di Pasquale C, D'Ippolito MR, Ditta S, Ducci E, Bell...",None,2024,10.3390/healthcare12181805,39337146,PMC11431503,Y,Y,2,"[{""availability"": ""Subscription required"", ""availabilityCode"": ""S"", ""documentStyle"": ""doi"", ""site"": ""DOI"", ""url"": ""https://doi.org/10.3390/healthcare12181805""}, {""availability"": ""Open access"", ""availabilityCode"": ""OA...","This study, employing an interim analysis, investigates the effects of the Dermamecum protocol, a structured educational and tailored approach that stratifies ostomy patients into risk paths (green, yellow, red) base...",https://doi.org/10.3390/healthcare12181805,abstract,4,"stratification of 226 patients, with significant differences in gender distribution, BMI categories, and stoma types across the paths. Results show an occurrence rate of PSCs of 5.9% in all risk paths (5.7% green pat...",0
1,Europe PMC,"""32032"" ""atopic dermatitis""",Outcomes of a Risk-Stratified Protocol for Preventing Peristomal Skin Complications in Patients with an Ostomy: A Cohort Study.,"Denti FC, Guerra E, Caroppo F, Abruzzese P, Alessi F, Barone F, Bernardino P, Bergamini M, Bernardo MC, Bosio G, Carp P, Cecconello M, Cerchier A, Croci F, Detti R, Dimitrova MM, Di Pasquale C, D'Ippolito MR, Ditta S...",None,2025,10.3390/nursrep15050179,40423212,PMC12114260,Y,Y,5,"[{""availability"": ""Subscription required"", ""availabilityCode"": ""S"", ""documentStyle"": ""doi"", ""site"": ""DOI"", ""url"": ""https://doi.org/10.3390/nursrep15050179""}, {""availability"": ""Open access"", ""availabilityCode"": ""OA"", ...","<h4>Background/objectives</h4>Peristomal skin complications (PSCs) are common among patients with ostomies, significantly impacting quality of life and increasing healthcare utilization. This study aimed to evaluate ...",https://doi.org/10.3390/nursrep15050179,abstract,4,Outcomes of a Risk-Stratified Protocol for Preventing Peristomal Skin Complications in Patients with an Ostomy: A Cohort Study. <h4>Background/objectives</h4>Peristomal skin complications (PSCs) are common among pati...,0
2,Europe PMC,"""32032"" ""atopic dermatitis""",The pathogen Vibrio alginolyticus H1 and its antagonist Pseudoalteromonas piscicida H2 associated with the health status of cuttlefish Sepia pharaonis,"Xu L, Jiang M, Peng R, Jiang X, Wang S, Han Q, Zhang W.",None,2024,None,None,PMC10901913,Y,Y,0,"[{""availability"": ""Free"", ""availabilityCode"": ""F"", ""documentStyle"": ""html"", ""site"": ""PubMedCentral"", ""url"": ""https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10901913/?tool=EBI""}, {""availability"": ""Free"", ""availabilityCo...",None,,metadata,,,0
3,Europe PMC,"""32032"" ""atopic dermatitis""",Agarwood Inhibits Histamine Release from Rat Mast Cells and Reduces Scratching Behavior in Mice: Effect of Agarwood on Histamine Release and Scratching Behavior.,"Inoue E, Shimizu Y, Masui R, Tsubonoya T, Hayakawa T, Sudoh K.",None,2016,10.3831/kpi.2016.19.025,27695633,PMC5043088,Y,Y,2,"[{""availability"": ""Subscription required"", ""availabilityCode"": ""S"", ""documentStyle"": ""doi"", ""site"": ""DOI"", ""url"": ""https://doi.org/10.3831/KPI.2016.19.025""}, {""availability"": ""Open access"", ""availabilityCode"": ""OA"", ...",<h4>Objectives</h4>This study was conducted to clarify the effects of agarwood on histamine release from mast cells in rats and on the scratching behaviors in mice.<h4>Methods</h4>Histamine release from rat mast cell...,https://doi.org/10.3831/kpi.2016.19.025,abst

## 9. ChEMBL alias and reference enrichment

ChEMBL can identify chemical synonyms, preferred names, max phase, indication rows, and mechanism rows. These aliases are added to the global alias table.


In [ ]:
def chembl_search_molecule(query):
    url = "https://www.ebi.ac.uk/chembl/api/data/molecule/search.json"
    data = safe_get(url, params={"q": query})
    return data.get("molecules", []) if isinstance(data, dict) else []


def chembl_drug_indications(chembl_id):
    url = "https://www.ebi.ac.uk/chembl/api/data/drug_indication.json"
    data = safe_get(url, params={"molecule_chembl_id": chembl_id, "limit": 100})
    return data.get("drug_indications", []) if isinstance(data, dict) else []


def chembl_mechanism(chembl_id):
    url = "https://www.ebi.ac.uk/chembl/api/data/mechanism.json"
    data = safe_get(url, params={"molecule_chembl_id": chembl_id, "limit": 100})
    return data.get("mechanisms", []) if isinstance(data, dict) else []


chembl_aliases_df = empty_df(["alias","alias_type","source","confidence","notes","molecule_chembl_id"])
chembl_evidence_df = empty_df(["source","query_used","molecule_chembl_id","pref_name","molecule_type","max_phase","synonyms","structure","cross_references","evidence_type","evidence_json","url"])

if USE_CHEMBL:
    rows, alias_rows = [], []
    for query in sorted(set([ASSET_NAME] + aliases_df["alias"].dropna().astype(str).tolist()))[:25]:
        molecules = chembl_search_molecule(query)
        for mol in molecules[:10]:
            chembl_id = mol.get("molecule_chembl_id")
            synonyms = []
            for syn in mol.get("molecule_synonyms") or []:
                if isinstance(syn, dict):
                    synonyms.append(first_nonempty(syn.get("molecule_synonym"), syn.get("synonyms")))
            pref = mol.get("pref_name")
            if pref:
                synonyms.append(pref)
            for syn in synonyms:
                if syn:
                    alias_rows.append({"alias": syn, "alias_type": "source_derived", "source": "ChEMBL", "confidence": 75, "notes": f"ChEMBL synonym for {chembl_id}", "molecule_chembl_id": chembl_id})
            base = {
                "source": "ChEMBL",
                "query_used": query,
                "molecule_chembl_id": chembl_id,
                "pref_name": pref,
                "molecule_type": mol.get("molecule_type"),
                "max_phase": mol.get("max_phase"),
                "synonyms": compact_join(synonyms),
                "structure": json.dumps(mol.get("molecule_structures") or {})[:1000],
                "cross_references": json.dumps(mol.get("cross_references") or [])[:1000],
                "url": f"https://www.ebi.ac.uk/chembl/compound_report_card/{chembl_id}/" if chembl_id else "",
            }
            rows.append({**base, "evidence_type": "molecule_search", "evidence_json": json.dumps(mol)[:2000]})
            if chembl_id:
                for item in chembl_drug_indications(chembl_id):
                    rows.append({**base, "evidence_type": "drug_indication", "evidence_json": json.dumps(item)[:2000]})
                for item in chembl_mechanism(chembl_id):
                    rows.append({**base, "evidence_type": "mechanism", "evidence_json": json.dumps(item)[:2000]})
        time.sleep(0.15)
    if alias_rows:
        chembl_aliases_df = pd.DataFrame(alias_rows).drop_duplicates(subset=["alias","source","molecule_chembl_id"])
    if rows:
        chembl_evidence_df = pd.DataFrame(rows).drop_duplicates(subset=["molecule_chembl_id","evidence_type","evidence_json"])
    aliases_df = pd.concat([aliases_df, chembl_aliases_df.drop(columns=["molecule_chembl_id"], errors="ignore")], ignore_index=True).drop_duplicates(subset=["alias","source"])
    log_source_status("ChEMBL", searched=True, records=len(chembl_evidence_df), reason="Molecule, indication, and mechanism enrichment")
else:
    log_source_status("ChEMBL", skipped=True, reason="USE_CHEMBL=False")

display(chembl_aliases_df.head())
display(chembl_evidence_df.head())


## 10. PubChem alias enrichment

PubChem PUG REST is used for CID, synonyms, and selected properties. Missing compounds are skipped cleanly.


In [ ]:
def pubchem_search_synonyms(query):
    name = quote_plus(str(query))
    cid_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{name}/cids/JSON"
    cid_data = safe_get(cid_url)
    cids = []
    if isinstance(cid_data, dict):
        cids = cid_data.get("IdentifierList", {}).get("CID", [])
    rows = []
    for cid in cids[:10]:
        syn_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/synonyms/JSON"
        prop_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/MolecularFormula,MolecularWeight,CanonicalSMILES,InChIKey/JSON"
        syn_data = safe_get(syn_url)
        prop_data = safe_get(prop_url)
        synonyms = []
        if isinstance(syn_data, dict):
            info = syn_data.get("InformationList", {}).get("Information", [])
            if info:
                synonyms = info[0].get("Synonym", []) or []
        props = {}
        if isinstance(prop_data, dict):
            plist = prop_data.get("PropertyTable", {}).get("Properties", [])
            props = plist[0] if plist else {}
        rows.append({"cid": cid, "query": query, "synonyms": synonyms, "properties": props})
        time.sleep(0.15)
    return rows


pubchem_aliases_df = empty_df(["alias","alias_type","source","confidence","notes","cid"])

if USE_PUBCHEM:
    alias_rows = []
    for query in sorted(set([ASSET_NAME] + aliases_df["alias"].dropna().astype(str).tolist()))[:20]:
        for result in pubchem_search_synonyms(query):
            cid = result["cid"]
            for syn in result["synonyms"][:80]:
                alias_rows.append({"alias": syn, "alias_type": "source_derived", "source": "PubChem", "confidence": 70, "notes": f"PubChem synonym for CID {cid}", "cid": cid})
    if alias_rows:
        pubchem_aliases_df = pd.DataFrame(alias_rows).drop_duplicates(subset=["alias","cid"])
    aliases_df = pd.concat([aliases_df, pubchem_aliases_df.drop(columns=["cid"], errors="ignore")], ignore_index=True).drop_duplicates(subset=["alias","source"])
    log_source_status("PubChem", searched=True, records=len(pubchem_aliases_df), reason="CID and synonym enrichment")
else:
    log_source_status("PubChem", skipped=True, reason="USE_PUBCHEM=False")

display(pubchem_aliases_df.head())


## 11. Crossref search

Crossref discovers DOI metadata, preprints, publisher pages, proceedings, and articles that may not surface through PubMed exact-alias search.


In [ ]:
def crossref_search(query, rows=20):
    url = "https://api.crossref.org/works"
    params = {"query": query, "rows": rows}
    data = safe_get(url, params=params)
    if isinstance(data, dict):
        return data.get("message", {}).get("items", [])
    return []


def crossref_date(item):
    for key in ["published-print", "published-online", "published", "created"]:
        parts = item.get(key, {}).get("date-parts", [])
        if parts and parts[0]:
            return "-".join(str(x) for x in parts[0])
    return ""


crossref_hits_df = empty_df(["source","query_used","doi","title","container_title","publisher","published_date","abstract","url","reference_count","is_referenced_by_count","license","match_location","alias_matched","match_context","confidence"])

if USE_CROSSREF:
    rows = []
    queries = query_bank_df[query_bank_df["priority"].le(3)]["query"].head(45).tolist() + list(KNOWN_DOIS)
    for query in tqdm(dict.fromkeys(queries), desc="Crossref"):
        for item in crossref_search(query, rows=20):
            title = compact_join(item.get("title"))
            abstract = re.sub("<[^>]+>", " ", item.get("abstract") or "")
            text_blob = " ".join([title, abstract, item.get("DOI") or ""])
            matches = find_alias_matches(text_blob, aliases_df["alias"].dropna().tolist(), window=220)
            if not matches:
                matches = [{"alias_matched": "", "match_context": ""}]
            for match in matches:
                rows.append({
                    "source": "Crossref",
                    "query_used": query,
                    "doi": item.get("DOI"),
                    "title": title,
                    "container_title": compact_join(item.get("container-title")),
                    "publisher": item.get("publisher"),
                    "published_date": crossref_date(item),
                    "abstract": abstract,
                    "url": item.get("URL") or (f"https://doi.org/{item.get('DOI')}" if item.get("DOI") else ""),
                    "reference_count": item.get("reference-count"),
                    "is_referenced_by_count": item.get("is-referenced-by-count"),
                    "license": json.dumps(item.get("license") or [])[:1000],
                    "match_location": "abstract" if match.get("alias_matched") and abstract else "metadata",
                    "alias_matched": match.get("alias_matched"),
                    "match_context": match.get("match_context"),
                    "confidence": 0,
                })
        time.sleep(0.15)
    if rows:
        crossref_hits_df = pd.DataFrame(rows).drop_duplicates(subset=["query_used","doi","title","alias_matched"])
    log_source_status("Crossref", searched=True, records=len(crossref_hits_df), reason=f"{len(queries)} queries")
else:
    log_source_status("Crossref", skipped=True, reason="USE_CROSSREF=False")

display(crossref_hits_df.head())


## 12. OpenAlex search

OpenAlex enriches publication metadata, open-access status, locations, authorships, concepts, citations, references, and related works.


In [ ]:
def openalex_search(query, per_page=25):
    url = "https://api.openalex.org/works"
    params = {"search": query, "per-page": per_page}
    if NCBI_EMAIL and NCBI_EMAIL != "your-email@example.com":
        params["mailto"] = NCBI_EMAIL
    data = safe_get(url, params=params)
    if isinstance(data, dict):
        return data.get("results", [])
    return []


openalex_hits_df = empty_df(["source","query_used","id","doi","title","publication_year","publication_date","authorships","primary_location","open_access","cited_by_count","concepts","referenced_works","related_works","url","match_location","alias_matched","match_context","confidence"])

if USE_OPENALEX:
    rows = []
    queries = query_bank_df[query_bank_df["priority"].le(3)]["query"].head(45).tolist()
    for query in tqdm(dict.fromkeys(queries), desc="OpenAlex"):
        for item in openalex_search(query, per_page=25):
            title = item.get("title") or ""
            text_blob = " ".join([title, item.get("doi") or ""])
            matches = find_alias_matches(text_blob, aliases_df["alias"].dropna().tolist(), window=200)
            if not matches:
                matches = [{"alias_matched": "", "match_context": ""}]
            authors = []
            for authorship in item.get("authorships") or []:
                author = authorship.get("author") or {}
                if author.get("display_name"):
                    authors.append(author.get("display_name"))
            concepts = [c.get("display_name") for c in item.get("concepts") or [] if c.get("display_name")]
            for match in matches:
                primary = item.get("primary_location") or {}
                rows.append({
                    "source": "OpenAlex",
                    "query_used": query,
                    "id": item.get("id"),
                    "doi": (item.get("doi") or "").replace("https://doi.org/", ""),
                    "title": title,
                    "publication_year": item.get("publication_year"),
                    "publication_date": item.get("publication_date"),
                    "authorships": compact_join(authors[:12]),
                    "primary_location": json.dumps(primary)[:1500],
                    "open_access": json.dumps(item.get("open_access") or {})[:1000],
                    "cited_by_count": item.get("cited_by_count"),
                    "concepts": compact_join(concepts[:20]),
                    "referenced_works": json.dumps(item.get("referenced_works") or [])[:1500],
                    "related_works": json.dumps(item.get("related_works") or [])[:1500],
                    "url": item.get("doi") or item.get("id"),
                    "match_location": "title" if match.get("alias_matched") else "metadata",
                    "alias_matched": match.get("alias_matched"),
                    "match_context": match.get("match_context"),
                    "confidence": 0,
                })
        time.sleep(0.15)
    if rows:
        openalex_hits_df = pd.DataFrame(rows).drop_duplicates(subset=["query_used","id","doi","title","alias_matched"])
    log_source_status("OpenAlex", searched=True, records=len(openalex_hits_df), reason=f"{len(queries)} queries")
else:
    log_source_status("OpenAlex", skipped=True, reason="USE_OPENALEX=False")

display(openalex_hits_df.head())


## 13. Semantic Scholar search

Semantic Scholar can add paper graph metadata, citation metrics, external IDs, and open-access PDF links. It uses an API key if available.


In [ ]:
def semantic_scholar_search(query, limit=20):
    url = "https://api.semanticscholar.org/graph/v1/paper/search"
    fields = "title,abstract,year,authors,venue,citationCount,influentialCitationCount,externalIds,url,openAccessPdf,references,citations"
    headers = {}
    if SEMANTIC_SCHOLAR_API_KEY:
        headers["x-api-key"] = SEMANTIC_SCHOLAR_API_KEY
    data = safe_get(url, params={"query": query, "limit": limit, "fields": fields}, headers=headers)
    if isinstance(data, dict):
        return data.get("data", [])
    return []


semantic_scholar_hits_df = empty_df(["source","query_used","paperId","title","abstract","year","authors","venue","citationCount","influentialCitationCount","externalIds","url","openAccessPdf","references","citations","match_location","alias_matched","match_context","confidence"])

if USE_SEMANTIC_SCHOLAR:
    rows = []
    queries = query_bank_df[query_bank_df["priority"].le(3)]["query"].head(35).tolist()
    for query in tqdm(dict.fromkeys(queries), desc="Semantic Scholar"):
        for item in semantic_scholar_search(query, limit=20):
            text_blob = " ".join([item.get("title") or "", item.get("abstract") or ""])
            matches = find_alias_matches(text_blob, aliases_df["alias"].dropna().tolist(), window=220)
            if not matches:
                matches = [{"alias_matched": "", "match_context": ""}]
            for match in matches:
                rows.append({
                    "source": "Semantic Scholar",
                    "query_used": query,
                    "paperId": item.get("paperId"),
                    "title": item.get("title"),
                    "abstract": item.get("abstract"),
                    "year": item.get("year"),
                    "authors": compact_join([a.get("name") for a in item.get("authors") or [] if isinstance(a, dict)]),
                    "venue": item.get("venue"),
                    "citationCount": item.get("citationCount"),
                    "influentialCitationCount": item.get("influentialCitationCount"),
                    "externalIds": json.dumps(item.get("externalIds") or {}),
                    "url": item.get("url"),
                    "openAccessPdf": json.dumps(item.get("openAccessPdf") or {}),
                    "references": json.dumps(item.get("references") or [])[:1500],
                    "citations": json.dumps(item.get("citations") or [])[:1500],
                    "match_location": "abstract" if match.get("alias_matched") and item.get("abstract") else ("title" if match.get("alias_matched") else "metadata"),
                    "alias_matched": match.get("alias_matched"),
                    "match_context": match.get("match_context"),
                    "confidence": 0,
                })
        time.sleep(0.3 if not SEMANTIC_SCHOLAR_API_KEY else 0.12)
    if rows:
        semantic_scholar_hits_df = pd.DataFrame(rows).drop_duplicates(subset=["query_used","paperId","title","alias_matched"])
    log_source_status("Semantic Scholar", searched=True, records=len(semantic_scholar_hits_df), reason=f"{len(queries)} queries; api_key={'yes' if SEMANTIC_SCHOLAR_API_KEY else 'no'}")
else:
    log_source_status("Semantic Scholar", skipped=True, reason="USE_SEMANTIC_SCHOLAR=False")

display(semantic_scholar_hits_df.head())


## 14. Exa / web OSINT discovery

When `EXA_API_KEY` is configured and `USE_EXA=True`, the notebook calls Exa REST. If disabled, the Exa query bank remains available for manual MCP or browser searches. This section targets web pages, PDFs, company pages, conference abstracts, investor decks, SEC filings, results pages, and discontinuation mentions.


In [ ]:
def exa_search(query, num_results=10):
    if not EXA_API_KEY:
        return []
    url = "https://api.exa.ai/search"
    headers = {"x-api-key": EXA_API_KEY, "Content-Type": "application/json"}
    payload = {
        "query": query,
        "numResults": num_results,
        "contents": {"text": {"maxCharacters": 1200}, "highlights": {"numSentences": 3}},
    }
    try:
        response = requests.post(url, headers=headers, json=payload, timeout=45)
        if not response.ok:
            print(f"[warn] Exa query failed HTTP {response.status_code}: {query}")
            return []
        return response.json().get("results", [])
    except Exception as exc:
        print(f"[warn] Exa query failed: {exc}")
        return []


def source_domain(url):
    try:
        return urlparse(url).netloc.lower().replace("www.", "")
    except Exception:
        return ""


exa_web_hits_df = empty_df(["source","query_used","title","url","published_date","author","text","highlights","summary","source_domain","match_location","alias_matched","match_context","confidence"])

if USE_EXA and EXA_API_KEY:
    rows = []
    exa_queries = exa_query_bank_df["query"].head(60).tolist()
    for query in tqdm(dict.fromkeys(exa_queries), desc="Exa"):
        for item in exa_search(query, num_results=10):
            text_blob = " ".join([
                item.get("title") or "",
                item.get("text") or "",
                " ".join(item.get("highlights") or []),
                item.get("summary") or "",
            ])
            matches = find_alias_matches(text_blob, aliases_df["alias"].dropna().tolist(), window=240)
            if not matches:
                matches = [{"alias_matched": "", "match_context": ""}]
            for match in matches:
                rows.append({
                    "source": "Exa",
                    "query_used": query,
                    "title": item.get("title"),
                    "url": item.get("url"),
                    "published_date": item.get("publishedDate"),
                    "author": item.get("author"),
                    "text": (item.get("text") or "")[:1500],
                    "highlights": json.dumps(item.get("highlights") or [])[:1500],
                    "summary": item.get("summary"),
                    "source_domain": source_domain(item.get("url") or ""),
                    "match_location": "web_snippet" if match.get("alias_matched") else "metadata",
                    "alias_matched": match.get("alias_matched"),
                    "match_context": match.get("match_context"),
                    "confidence": 0,
                })
        time.sleep(0.2)
    if rows:
        exa_web_hits_df = pd.DataFrame(rows).drop_duplicates(subset=["query_used","url","alias_matched"])
    log_source_status("Exa", searched=True, records=len(exa_web_hits_df), reason=f"{len(exa_queries)} web OSINT queries")
else:
    reason = "USE_EXA=False" if not USE_EXA else "EXA_API_KEY missing"
    log_source_status("Exa", skipped=True, reason=reason)
    print(f"[info] Exa skipped: {reason}. Use exa_query_bank_df for manual MCP/web searches.")

display(exa_query_bank_df.head(30))
display(exa_web_hits_df.head())


## 15. PDF and webpage fetcher

This section optionally fetches user-provided URLs, open-access full text URLs, Exa URLs, and PDF links. It does not bypass paywalls. PDF text extraction uses PyMuPDF if installed and stores only short snippets around matches.


In [ ]:
def fetch_webpage_text(url):
    html = safe_get(url, headers={"Accept": "text/html,*/*"}, timeout=35)
    if not isinstance(html, str):
        return {"text": "", "title": "", "needs_manual_review": True}
    if BeautifulSoup is None:
        return {"text": re.sub(r"<[^>]+>", " ", html), "title": "", "needs_manual_review": False}
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    title = soup.title.get_text(" ", strip=True) if soup.title else ""
    text = soup.get_text(" ", strip=True)
    return {"text": re.sub(r"\s+", " ", text), "title": title, "needs_manual_review": False}


def fetch_pdf_text(url):
    if fitz is None:
        return [{"page": None, "text": "", "needs_manual_review": True, "error": "PyMuPDF/fitz not installed"}]
    try:
        response = requests.get(url, headers={"User-Agent": "MindReaderBiotechDiscoveryNotebook/1.0"}, timeout=45)
        if not response.ok:
            return [{"page": None, "text": "", "needs_manual_review": True, "error": f"HTTP {response.status_code}"}]
        pdf_path = OUTPUT_DIR / ("tmp_" + hashlib.sha1(url.encode()).hexdigest() + ".pdf")
        pdf_path.write_bytes(response.content)
        rows = []
        with fitz.open(pdf_path) as doc:
            for page_number, page in enumerate(doc, start=1):
                rows.append({"page": page_number, "text": page.get_text("text"), "needs_manual_review": False, "error": ""})
        try:
            pdf_path.unlink()
        except Exception:
            pass
        return rows
    except Exception as exc:
        return [{"page": None, "text": "", "needs_manual_review": True, "error": str(exc)}]


def candidate_urls_for_fetch():
    urls = []
    urls.extend(KNOWN_URLS)
    if "url" in exa_web_hits_df.columns:
        urls.extend(exa_web_hits_df["url"].dropna().tolist())
    for df in [europe_pmc_hits_df, semantic_scholar_hits_df, crossref_hits_df, openalex_hits_df]:
        if "url" in df.columns:
            urls.extend(df["url"].dropna().tolist())
    return [u for u in dict.fromkeys(urls) if isinstance(u, str) and u.startswith("http")]


pdf_body_hits_df = empty_df(["source","url","title","page","alias_matched","match_context","match_location","needs_manual_review","confidence"])

if USE_PDF_FETCH:
    rows = []
    aliases = aliases_df["alias"].dropna().tolist()
    urls = candidate_urls_for_fetch()[:80]
    for url in tqdm(urls, desc="Fetch web/PDF"):
        is_pdf = ".pdf" in url.lower() or url.lower().split("?")[0].endswith(".pdf")
        if is_pdf:
            page_rows = fetch_pdf_text(url)
            for page_row in page_rows:
                matches = find_alias_matches(page_row.get("text",""), aliases, window=240)
                if not matches and page_row.get("needs_manual_review"):
                    rows.append({"source": "PDF", "url": url, "title": "", "page": page_row.get("page"), "alias_matched": "", "match_context": page_row.get("error",""), "match_location": "pdf_body", "needs_manual_review": True, "confidence": 0})
                for match in matches:
                    rows.append({"source": "PDF", "url": url, "title": "", "page": page_row.get("page"), "alias_matched": match.get("alias_matched"), "match_context": match.get("match_context"), "match_location": "pdf_body", "needs_manual_review": page_row.get("needs_manual_review", False), "confidence": 0})
        else:
            page = fetch_webpage_text(url)
            matches = find_alias_matches(page.get("text",""), aliases, window=240)
            for match in matches:
                rows.append({"source": "Webpage", "url": url, "title": page.get("title"), "page": None, "alias_matched": match.get("alias_matched"), "match_context": match.get("match_context"), "match_location": "body", "needs_manual_review": page.get("needs_manual_review", False), "confidence": 0})
        time.sleep(0.25)
    if rows:
        pdf_body_hits_df = pd.DataFrame(rows).drop_duplicates(subset=["url","page","alias_matched","match_context"])
    log_source_status("PDF/web body fetch", searched=True, records=len(pdf_body_hits_df), reason=f"{len(urls)} URLs attempted; PyMuPDF={'yes' if fitz else 'no'}")
else:
    log_source_status("PDF/web body fetch", skipped=True, reason="USE_PDF_FETCH=False")

display(pdf_body_hits_df.head())


## 16. Match location classifier

Location classification standardizes evidence rows for scoring and missed-by-PubMed analysis.


In [ ]:
def classify_match_location(record):
    text = " ".join(str(record.get(k, "")) for k in ["match_location", "section", "source", "url"]).lower()
    if "title" in text:
        return "title"
    if "abstract" in text:
        return "abstract"
    if "table" in text:
        return "table"
    if "fig" in text or "caption" in text:
        return "figure_caption"
    if "reference" in text:
        return "reference"
    if "pdf" in text:
        return "pdf_body"
    if "snippet" in text or "exa" in text:
        return "web_snippet"
    if "body" in text or "webpage" in text:
        return "body"
    if "metadata" in text:
        return "metadata"
    return "unknown"


## 17. Confidence scoring

The score rewards direct asset mentions, stronger match locations, sponsor/disease/target context, persistent identifiers, and user-provided inputs. Weak sponsor+disease hits without direct asset aliases are capped and routed to manual review.


In [ ]:
def evidence_bucket(score):
    if score >= 85:
        return "confirmed direct evidence"
    if score >= 65:
        return "strong likely evidence"
    if score >= 45:
        return "related / review manually"
    return "weak / probably noise"


def score_document_hit(row):
    loc = classify_match_location(row)
    score = 0
    if loc == "title":
        score += 40
    elif loc == "abstract":
        score += 35
    elif loc in {"body", "table", "figure_caption"}:
        score += 30
    elif loc == "pdf_body":
        score += 25
    elif loc == "web_snippet":
        score += 15
    elif loc == "metadata":
        score += 8

    text = " ".join(str(row.get(k, "")) for k in ["title","abstract","match_context","query_used","journal","journal_or_site","source","url"])
    alias = str(row.get("alias_matched", "") or "").strip()
    direct_alias = bool(alias)
    if direct_alias:
        score += 30
    if SPONSOR and re.search(re.escape(SPONSOR), text, flags=re.IGNORECASE):
        score += 15
    if DISEASE and re.search(re.escape(DISEASE), text, flags=re.IGNORECASE):
        score += 15
    if any(t and re.search(re.escape(str(t)), text, flags=re.IGNORECASE) for t in TARGETS):
        score += 10
    if first_nonempty(row.get("pmid"), row.get("doi"), row.get("pmcid")):
        score += 10
    if str(row.get("url","")) in KNOWN_URLS or str(row.get("pmid","")) in [str(x) for x in KNOWN_PMIDS] or str(row.get("doi","")).lower() in [str(x).lower() for x in KNOWN_DOIS]:
        score += 20
    if not direct_alias and SPONSOR and DISEASE and re.search(re.escape(SPONSOR), text, flags=re.IGNORECASE) and re.search(re.escape(DISEASE), text, flags=re.IGNORECASE):
        score = min(score, 55)
    score = max(0, min(100, int(score)))
    needs_manual_review = (not direct_alias) or score < 65
    return pd.Series({"confidence": score, "evidence_bucket": evidence_bucket(score), "needs_manual_review": needs_manual_review})


## 18. Deduplication

All source tables are normalized into a common evidence schema. Deduplication prioritizes DOI, PMID, PMCID, normalized URL, then normalized title hash, while preserving all queries, sources, aliases, match locations, identifiers, and best confidence.


In [ ]:
def normalize_title(title):
    title = normalize_ascii(str(title or "")).lower()
    title = re.sub(r"[^a-z0-9]+", " ", title)
    return re.sub(r"\s+", " ", title).strip()


def normalize_url(url):
    if not url:
        return ""
    try:
        parsed = urlparse(str(url).strip())
        parsed = parsed._replace(fragment="", query="")
        netloc = parsed.netloc.lower().replace("www.", "")
        path = parsed.path.rstrip("/")
        return urlunparse((parsed.scheme.lower(), netloc, path, "", "", ""))
    except Exception:
        return str(url).strip().lower()


def common_rows_from_df(df, source_name, mappings):
    rows = []
    if df is None or df.empty:
        return rows
    for _, r in df.iterrows():
        row = {
            "source": source_name,
            "source_types": source_name,
            "query_used": r.get(mappings.get("query_used","query_used"), ""),
            "title": r.get(mappings.get("title","title"), ""),
            "url": r.get(mappings.get("url","url"), ""),
            "doi": r.get(mappings.get("doi","doi"), ""),
            "pmid": r.get(mappings.get("pmid","pmid"), ""),
            "pmcid": r.get(mappings.get("pmcid","pmcid"), ""),
            "pub_date": r.get(mappings.get("pub_date","pub_date"), ""),
            "journal_or_site": r.get(mappings.get("journal_or_site","journal"), r.get("journalTitle", r.get("container_title", r.get("venue", r.get("source_domain",""))))),
            "alias_matched": r.get("alias_matched", ""),
            "match_location": classify_match_location(r.to_dict()),
            "match_context": r.get("match_context", ""),
            "confidence": r.get("confidence", 0),
            "raw_source": source_name,
        }
        rows.append(row)
    return rows


common_rows = []
common_rows += common_rows_from_df(pubmed_hits_df, "PubMed", {"journal_or_site": "journal"})
common_rows += common_rows_from_df(pmc_fulltext_hits_df, "PMC BioC", {})
common_rows += common_rows_from_df(europe_pmc_hits_df, "Europe PMC", {"journal_or_site": "journalTitle", "pub_date": "pubYear"})
common_rows += common_rows_from_df(crossref_hits_df, "Crossref", {"pub_date": "published_date", "journal_or_site": "container_title"})
common_rows += common_rows_from_df(openalex_hits_df, "OpenAlex", {"pub_date": "publication_date", "journal_or_site": "primary_location"})
common_rows += common_rows_from_df(semantic_scholar_hits_df, "Semantic Scholar", {"pub_date": "year", "journal_or_site": "venue"})
common_rows += common_rows_from_df(exa_web_hits_df, "Exa", {"pub_date": "published_date", "journal_or_site": "source_domain"})
common_rows += common_rows_from_df(pdf_body_hits_df, "PDF/Web body", {})
common_rows += [{
    "source": "ClinicalTrials.gov",
    "source_types": "ClinicalTrials.gov",
    "query_used": r.get("query_used", ""),
    "title": first_nonempty(r.get("brief_title"), r.get("official_title")),
    "url": r.get("url", ""),
    "doi": "",
    "pmid": "",
    "pmcid": "",
    "pub_date": r.get("start_date", ""),
    "journal_or_site": "ClinicalTrials.gov",
    "alias_matched": ASSET_NAME if re.search(re.escape(ASSET_NAME.replace("-", "")), str(r.get("interventions","")).replace("-", ""), re.IGNORECASE) else "",
    "match_location": "metadata",
    "match_context": compact_join([r.get("brief_title"), r.get("interventions"), r.get("conditions")]),
    "confidence": 0,
    "raw_source": "ClinicalTrials.gov",
} for _, r in ctgov_trials_df.iterrows()]

all_hits_df = pd.DataFrame(common_rows) if common_rows else empty_df(["source","source_types","query_used","title","url","doi","pmid","pmcid","pub_date","journal_or_site","alias_matched","match_location","match_context","confidence","raw_source"])
if not all_hits_df.empty:
    scored = all_hits_df.apply(score_document_hit, axis=1)
    all_hits_df[["confidence","evidence_bucket","needs_manual_review"]] = scored


def dedupe_key(row):
    doi = str(row.get("doi","")).lower().replace("https://doi.org/","").strip()
    pmid = str(row.get("pmid","")).strip()
    pmcid = str(row.get("pmcid","")).upper().strip()
    url = normalize_url(row.get("url",""))
    title_norm = normalize_title(row.get("title",""))
    if doi and doi.lower() != "nan":
        return "doi:" + doi
    if pmid and pmid.lower() != "nan":
        return "pmid:" + pmid
    if pmcid and pmcid.lower() != "nan":
        return "pmcid:" + pmcid
    if url:
        return "url:" + url
    if title_norm:
        return "title:" + hashlib.sha1(title_norm.encode()).hexdigest()
    return "row:" + hashlib.sha1(json.dumps(row.to_dict(), sort_keys=True, default=str).encode()).hexdigest()


deduped_document_hits_df = empty_df(["evidence_id","title","source_types","best_source","url","doi","pmid","pmcid","pub_date","journal_or_site","aliases_matched","match_locations","best_match_context","confidence","evidence_bucket","needs_manual_review","queries_used"])
if not all_hits_df.empty:
    all_hits_df["dedupe_key"] = all_hits_df.apply(dedupe_key, axis=1)
    grouped_rows = []
    for idx, (key, group) in enumerate(all_hits_df.groupby("dedupe_key"), start=1):
        group = group.sort_values("confidence", ascending=False)
        best = group.iloc[0]
        grouped_rows.append({
            "evidence_id": f"EVID-{idx:05d}",
            "title": first_nonempty(*group["title"].tolist()),
            "source_types": compact_join(group["source"].dropna().unique()),
            "best_source": best.get("source"),
            "url": first_nonempty(*group["url"].tolist()),
            "doi": first_nonempty(*group["doi"].tolist()),
            "pmid": first_nonempty(*group["pmid"].tolist()),
            "pmcid": first_nonempty(*group["pmcid"].tolist()),
            "pub_date": first_nonempty(*group["pub_date"].tolist()),
            "journal_or_site": first_nonempty(*group["journal_or_site"].tolist()),
            "aliases_matched": compact_join(group["alias_matched"].dropna().unique()),
            "match_locations": compact_join(group["match_location"].dropna().unique()),
            "best_match_context": first_nonempty(*group["match_context"].tolist()),
            "confidence": int(group["confidence"].max()),
            "evidence_bucket": evidence_bucket(int(group["confidence"].max())),
            "needs_manual_review": bool(group["needs_manual_review"].any()),
            "queries_used": compact_join(group["query_used"].dropna().unique()),
        })
    deduped_document_hits_df = pd.DataFrame(grouped_rows).sort_values(["confidence","title"], ascending=[False, True]).reset_index(drop=True)

display(deduped_document_hits_df.head(25))


## 19. Missed-by-PubMed analysis

This table highlights records where an asset alias appears in body/PDF/web/full text but exact PubMed alias searches did not retrieve the document. These are the cases that explain why PubMed can find only a small number of BEN-2293 articles while full-text/PDF/web discovery can find more.


In [ ]:
def reason_pubmed_missed(row):
    locs = str(row.get("match_locations","")).lower()
    if "table" in locs:
        return "asset only appears in table"
    if "figure_caption" in locs:
        return "asset only appears in figure caption"
    if "pdf_body" in locs:
        return "asset only appears in PDF"
    if "body" in locs:
        return "asset only appears in body text"
    if not first_nonempty(row.get("pmid"), ""):
        return "no PMID"
    if any(x in str(row.get("source_types","")).lower() for x in ["exa", "pdf", "web", "clinicaltrials"]):
        return "conference / company / investor deck, registry, or web material not necessarily PubMed-indexed"
    return "metadata uses different alias or PubMed exact alias query did not retrieve it"


pubmed_pmids = set(pubmed_hits_df.get("pmid", pd.Series(dtype=str)).dropna().astype(str))
pubmed_dois = set(pubmed_hits_df.get("doi", pd.Series(dtype=str)).dropna().astype(str).str.lower())
miss_rows = []
for _, row in deduped_document_hits_df.iterrows():
    direct_alias = bool(str(row.get("aliases_matched","")).strip())
    full_textish = any(loc in str(row.get("match_locations","")).lower() for loc in ["body","table","figure_caption","pdf_body","web_snippet"])
    in_pubmed = (str(row.get("pmid","")) in pubmed_pmids and str(row.get("pmid",""))) or (str(row.get("doi","")).lower() in pubmed_dois and str(row.get("doi","")))
    if direct_alias and full_textish and not in_pubmed:
        miss_rows.append({
            "title": row.get("title"),
            "source": row.get("source_types"),
            "url": row.get("url"),
            "doi": row.get("doi"),
            "pmid": row.get("pmid"),
            "pmcid": row.get("pmcid"),
            "alias_matched": row.get("aliases_matched"),
            "match_location": row.get("match_locations"),
            "reason_pubmed_missed": reason_pubmed_missed(row),
            "match_context": row.get("best_match_context"),
            "confidence": row.get("confidence"),
        })

missed_by_pubmed_df = pd.DataFrame(miss_rows) if miss_rows else empty_df(["title","source","url","doi","pmid","pmcid","alias_matched","match_location","reason_pubmed_missed","match_context","confidence"])
display(missed_by_pubmed_df.head(25))


## 20. Final asset dossier

The dossier is generated from source rows. Every conclusion should be traceable to one or more table records above.


In [ ]:
def make_markdown_table(df, columns=None, max_rows=10):
    if df is None or df.empty:
        return "_No rows found._"
    view = df.copy()
    if columns:
        view = view[[c for c in columns if c in view.columns]]
    return view.head(max_rows).to_markdown(index=False)


source_coverage_df = pd.DataFrame(SOURCE_STATUS).drop_duplicates(subset=["source","reason"], keep="last")
top_confirmed = deduped_document_hits_df[deduped_document_hits_df["confidence"].ge(65)].head(15) if not deduped_document_hits_df.empty else deduped_document_hits_df
manual_queue_df = deduped_document_hits_df[deduped_document_hits_df["needs_manual_review"].eq(True)].head(25) if not deduped_document_hits_df.empty else deduped_document_hits_df

dossier_md = f"""
# Asset Discovery Dossier: {ASSET_NAME}

Generated: {datetime.utcnow().isoformat(timespec='seconds')}Z

## 1. Asset identity

- Asset: {ASSET_NAME}
- Sponsor/company: {SPONSOR}
- Disease/indication: {DISEASE}
- Targets/mechanism terms: {compact_join(TARGETS)}
- Known NCT IDs supplied: {compact_join(NCT_IDS) or 'None supplied'}

## 2. Known aliases

{make_markdown_table(aliases_df, ['alias','alias_type','source','confidence','notes'], 20)}

## 3. Trial registry evidence

{make_markdown_table(ctgov_trials_df, ['nct_id','brief_title','sponsor','conditions','interventions','phase','status','url'], 10)}

## 4. Direct article evidence

{make_markdown_table(top_confirmed, ['evidence_id','title','source_types','doi','pmid','pmcid','confidence','evidence_bucket','url'], 15)}

## 5. Full-text/body-text evidence

{make_markdown_table(pmc_fulltext_hits_df, ['pmid','pmcid','section','match_location','alias_matched','match_context','url'], 10)}

## 6. Company / OSINT evidence

{make_markdown_table(exa_web_hits_df, ['title','url','source_domain','alias_matched','match_context','confidence'], 10)}

## 7. Chemical database aliases

{make_markdown_table(pd.concat([chembl_aliases_df, pubchem_aliases_df], ignore_index=True), ['alias','source','confidence','notes'], 20)}

## 8. Conference / PDF / investor material

{make_markdown_table(pdf_body_hits_df, ['url','title','page','alias_matched','match_context','needs_manual_review'], 10)}

## 9. Missed-by-PubMed documents

{make_markdown_table(missed_by_pubmed_df, ['title','source','url','alias_matched','match_location','reason_pubmed_missed','confidence'], 20)}

## 10. Manual review queue

{make_markdown_table(manual_queue_df, ['evidence_id','title','source_types','confidence','evidence_bucket','url'], 25)}

## 11. Limitations

We retrieve all discoverable public, open-access, and user-provided documents from configured sources. Paywalled, private, unpublished, removed, or inaccessible material may not be retrievable.

PubMed mainly catches title / abstract / metadata. It may miss body-text, table, figure-caption, supplementary, PDF-only, company, conference, and investor-deck evidence. Scores are prioritization aids, not regulatory conclusions.

## 12. Recommended next searches

- Run high-priority Exa query bank rows manually or with `EXA_API_KEY`.
- Add known company domain to `COMPANY_DOMAIN` and rerun.
- Add user-provided PMIDs, DOIs, URLs, and NCT IDs from internal notes.
- Review `missed_by_pubmed_df` and `manual_queue_df` before making claims.
- For oncology assets, prioritize ASCO, ESMO, AACR, SITC, EHA, ASH, SEC, and company investor pages.
- For autoimmune assets, prioritize ACR, EULAR, EADV, DDW, UEG, ADA, ENDO, registry result pages, and company PDFs.

## Source coverage

{make_markdown_table(source_coverage_df, ['source','searched','skipped','reason','records'], 50)}
"""

print(dossier_md[:6000])


## 21. Exports

All requested CSVs and a JSON dossier are written to `discovery_outputs/`.


In [ ]:
aliases_df.to_csv(OUTPUT_DIR / "aliases.csv", index=False)
query_bank_df.to_csv(OUTPUT_DIR / "query_bank.csv", index=False)
ctgov_trials_df.to_csv(OUTPUT_DIR / "ctgov_trials.csv", index=False)
pubmed_hits_df.to_csv(OUTPUT_DIR / "pubmed_hits.csv", index=False)
pmc_fulltext_hits_df.to_csv(OUTPUT_DIR / "pmc_fulltext_hits.csv", index=False)
europe_pmc_hits_df.to_csv(OUTPUT_DIR / "europe_pmc_hits.csv", index=False)
chembl_aliases_df.to_csv(OUTPUT_DIR / "chembl_aliases.csv", index=False)
crossref_hits_df.to_csv(OUTPUT_DIR / "crossref_hits.csv", index=False)
openalex_hits_df.to_csv(OUTPUT_DIR / "openalex_hits.csv", index=False)
semantic_scholar_hits_df.to_csv(OUTPUT_DIR / "semantic_scholar_hits.csv", index=False)
exa_web_hits_df.to_csv(OUTPUT_DIR / "exa_web_hits.csv", index=False)
pdf_body_hits_df.to_csv(OUTPUT_DIR / "pdf_body_hits.csv", index=False)
deduped_document_hits_df.to_csv(OUTPUT_DIR / "deduped_document_hits.csv", index=False)
missed_by_pubmed_df.to_csv(OUTPUT_DIR / "missed_by_pubmed.csv", index=False)
source_coverage_df.to_csv(OUTPUT_DIR / "source_coverage.csv", index=False)

asset_dossier = {
    "asset_identity": {
        "asset_name": ASSET_NAME,
        "sponsor": SPONSOR,
        "disease": DISEASE,
        "nct_ids": NCT_IDS,
        "targets": TARGETS,
        "known_pmids": KNOWN_PMIDS,
        "known_dois": KNOWN_DOIS,
        "known_urls": KNOWN_URLS,
        "run_started_at": RUN_STARTED_AT,
        "run_finished_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    },
    "retrieval_limitation": "We retrieve all discoverable public, open-access, and user-provided documents from configured sources. Paywalled, private, unpublished, removed, or inaccessible material may not be retrievable.",
    "aliases": aliases_df.to_dict(orient="records"),
    "top_documents": deduped_document_hits_df.head(50).to_dict(orient="records"),
    "missed_by_pubmed": missed_by_pubmed_df.to_dict(orient="records"),
    "source_coverage": source_coverage_df.to_dict(orient="records"),
    "dossier_markdown": dossier_md,
}

with open(OUTPUT_DIR / "asset_dossier.json", "w") as f:
    json.dump(asset_dossier, f, indent=2, default=str)

print("Exported files:")
for path in sorted(OUTPUT_DIR.glob("*")):
    print("-", path)


## 22. Quality checks / test cases

These tests exercise matching, classification, scoring, dedupe, and missed-by-PubMed logic for edge cases. They are lightweight and do not require network access.


In [ ]:
test_cases = [
    {"case": "Exact title match", "title": "BEN-2293 improves signs of disease", "abstract": "", "body": "", "loc": "title", "url": "https://example.org/1", "doi": "10.1/a"},
    {"case": "Abstract-only match", "title": "A trial", "abstract": "BEN-2293 was studied.", "body": "", "loc": "abstract", "url": "https://example.org/2", "doi": "10.1/b"},
    {"case": "Body-only match", "title": "A trial", "abstract": "", "body": "The body mentions BEN-2293.", "loc": "body", "url": "https://example.org/3", "doi": ""},
    {"case": "Table-only match", "title": "A trial", "abstract": "", "body": "Table 1: BEN-2293 dose.", "loc": "table", "url": "https://example.org/4", "doi": ""},
    {"case": "Figure-caption match", "title": "A trial", "abstract": "", "body": "Figure 2. BEN-2293 response.", "loc": "figure_caption", "url": "https://example.org/5", "doi": ""},
    {"case": "PDF-only match", "title": "Deck", "abstract": "", "body": "BEN-2293 investor slide.", "loc": "pdf_body", "url": "https://example.org/deck.pdf", "doi": ""},
    {"case": "Unicode hyphen match", "title": "BEN–2293 study", "abstract": "", "body": "", "loc": "title", "url": "https://example.org/6", "doi": ""},
    {"case": "No-hyphen variant match", "title": "BEN2293 study", "abstract": "", "body": "", "loc": "title", "url": "https://example.org/7", "doi": ""},
    {"case": "Space variant match", "title": "BEN 2293 study", "abstract": "", "body": "", "loc": "title", "url": "https://example.org/8", "doi": ""},
    {"case": "Line-break variant match", "title": "BEN-\\n2293 study", "abstract": "", "body": "", "loc": "title", "url": "https://example.org/9", "doi": ""},
    {"case": "NCT-only discovery", "title": "NCT04737304 registry", "abstract": "", "body": "", "loc": "metadata", "url": "https://clinicaltrials.gov/study/NCT04737304", "doi": ""},
    {"case": "Sponsor+disease discovery", "title": f"{SPONSOR} {DISEASE} trial", "abstract": "", "body": "", "loc": "metadata", "url": "https://example.org/10", "doi": ""},
    {"case": "Target+disease discovery", "title": f"TrkA {DISEASE}", "abstract": "", "body": "", "loc": "metadata", "url": "https://example.org/11", "doi": ""},
    {"case": "No PMID document", "title": "Company page BEN-2293", "abstract": "", "body": "", "loc": "web_snippet", "url": "https://example.org/company", "doi": ""},
    {"case": "Paywalled / inaccessible document", "title": "Paywalled BEN-2293", "abstract": "", "body": "", "loc": "metadata", "url": "https://example.org/paywall", "doi": ""},
    {"case": "User-provided DOI", "title": "Known DOI BEN-2293", "abstract": "", "body": "", "loc": "metadata", "url": "https://doi.org/10.1/user", "doi": "10.1/user"},
    {"case": "User-provided URL", "title": "Known URL BEN-2293", "abstract": "", "body": "", "loc": "web_snippet", "url": "https://example.org/user", "doi": ""},
    {"case": "Duplicate DOI from multiple sources", "title": "Dup BEN-2293", "abstract": "", "body": "", "loc": "title", "url": "https://example.org/dup1", "doi": "10.1/dup"},
    {"case": "Duplicate title with no DOI", "title": "Same BEN-2293 title", "abstract": "", "body": "", "loc": "title", "url": "", "doi": ""},
    {"case": "Weak OSINT hit requiring manual review", "title": f"{SPONSOR} {DISEASE}", "abstract": "", "body": "", "loc": "web_snippet", "url": "https://example.org/weak", "doi": ""},
]

test_rows = []
aliases_for_tests = generate_deterministic_aliases(ASSET_NAME)
for t in test_cases:
    text = " ".join([t["title"], t["abstract"], t["body"]])
    matches = find_alias_matches(text, aliases_for_tests, window=120)
    alias = matches[0]["alias_matched"] if matches else ""
    context = matches[0]["match_context"] if matches else ""
    row = {
        "source": "test",
        "title": t["title"],
        "abstract": t["abstract"],
        "url": t["url"],
        "doi": t["doi"],
        "pmid": "",
        "pmcid": "",
        "match_location": t["loc"],
        "alias_matched": alias,
        "match_context": context,
        "query_used": t["case"],
    }
    score = score_document_hit(row)
    test_rows.append({**t, **row, **score.to_dict(), "matched": bool(alias)})

quality_checks_df = pd.DataFrame(test_rows)
display(quality_checks_df[["case","matched","match_location","alias_matched","confidence","evidence_bucket","needs_manual_review"]])

assert quality_checks_df.loc[quality_checks_df["case"].eq("Unicode hyphen match"), "matched"].iloc[0]
assert quality_checks_df.loc[quality_checks_df["case"].eq("No-hyphen variant match"), "matched"].iloc[0]
assert quality_checks_df.loc[quality_checks_df["case"].eq("Space variant match"), "matched"].iloc[0]
assert quality_checks_df.loc[quality_checks_df["case"].eq("Line-break variant match"), "matched"].iloc[0]
assert quality_checks_df.loc[quality_checks_df["case"].eq("Weak OSINT hit requiring manual review"), "needs_manual_review"].iloc[0]

print("Quality checks passed.")


## 23. Important rules and final source audit

This notebook follows the requested constraints: no Pydantic, no backend, no database, no FastAPI, no paid scraping, no paywall bypassing, short snippets only, source identifiers preserved, traceable conclusions, graceful API failures, Exa-disabled utility, modular functions, markdown explanations, pandas display tables, and explicit source searched/skipped reporting.


In [ ]:
rules_df = pd.DataFrame([
    {"rule": "No Pydantic", "status": "followed"},
    {"rule": "No backend", "status": "followed"},
    {"rule": "No database", "status": "followed"},
    {"rule": "No FastAPI", "status": "followed"},
    {"rule": "No paid scraping / paywall bypass", "status": "followed"},
    {"rule": "Store snippets only", "status": "followed"},
    {"rule": "Preserve source URL / DOI / PMID / PMCID when available", "status": "followed"},
    {"rule": "Conclusions traceable to source rows", "status": "followed"},
    {"rule": "Notebook useful if Exa disabled", "status": "followed"},
])
display(rules_df)
display(source_coverage_df[["source","searched","skipped","reason","records"]])
